# 第 66 课：从 PPO 机制走向 LLM 后训练
## Token 级 RLVR、组相对优势与工程预算

经过第 61–65 课的 Actor/Critic、GAE、PPO 稳定性和消融，本课把相同的基本问题迁回语言模型：**一个回答由多枚 token 组成，真实奖励常常只在结束时确定。**

从完整链路出发：

```text
提示 → 当前模型逐token生成 → EOS / 固定任务上限
  → 规则验证器打分 → 同题G条回答计算组优势
  → 逐token的当前/旧概率比、裁剪与参考KL
  → 更新LoRA → 丢弃旧批次 → 重新采样
```

此外独立训练一个小**价值头**，观察它与组相对基线的差别，使用终止与外部截断的 GAE 核对最后一步是否该 bootstrap。最后制作 GPU 内存**下界估计**，不要把算术估计冒充3060实测。

**运行边界**：Python 3.10+、已安装 PyTorch、CPU 即可。默认没有模型下载、没有外部 API、没有真实用户数据。训练的是经短SFT的随机小型字符 Transformer，**不是预训练LLM、没有真正的数学推理能力实验，也不是完整TRL GRPO复现**。配套脚本和本 Notebook 都是独立完整文件。

## 0. 学习路线与环境

两次学习最合适：第一次学 EOS、采样、mask、组优势与归约；第二次学价值、GAE、真实训练、内存预算及独立练习。可折叠实现代码需要执行，但初读无需逐行抄写。

课程固定四个归一化/奖励缩放条件：

| 实验分支 | token 代理目标分母 | 组内奖励标准化 |
|---|---|---|
| `grpo` | 每条回答长度，之后按回答数平均 | 是 |
| `dapo` | 本批所有有效 token 数 | 是 |
| `dr_grpo` | 全局固定 `N×max_completion` | 是 |
| `dr_noscale` | 同一固定分母 | 否 |

前三者**只改归约分母**，第三与第四者**只改奖励缩放**。这样比同时改变两者更便于归因。它们是本课受控实现；实践框架有额外损失、样本过滤、同步与配置，不声称与框架的每项细节一致。推荐与默认值也随框架版本变化。[TRL GRPO v0.28.0](https://huggingface.co/docs/trl/v0.28.0/en/grpo_trainer)

### 支持实现：环境、版本、配置、哈希与保存

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""第66课：从PPO到LLM后训练——token级RLVR、组优势与工程预算。

独立 Python 3.10+ / PyTorch；默认 CPU 离线、人工可验证的字符复制任务。
实测模型是随机初始化后经短暂SFT的微型因果Transformer，不是预训练LLM。
教学内容：EOS/长度封顶、逐token采样和mask、GRPO/DAPO/Dr.GRPO三种归约、
参考KL、一个单独的value head及终止/外部截断的GAE、显存下界估算。
这些是明确配置的教学变体；并非复现TRL或证明大模型方法优劣。
"""
from __future__ import annotations

import argparse
import copy
import hashlib
import itertools
import json
import math
import platform
import random
import tempfile
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ImportError as exc:
    raise SystemExit("需要已有PyTorch环境，请使用同一内核；本课不自动安装依赖") from exc

VERSION = "lesson66-token-rlvr-grpo-v1"
SOURCES = {
    "grpo_paper": "https://arxiv.org/abs/2402.03300",
    "trl_grpo_v028": "https://huggingface.co/docs/trl/v0.28.0/en/grpo_trainer",
    "trl_ppo": "https://huggingface.co/docs/trl/main/ppo_trainer",
    "ppo_paper": "https://arxiv.org/abs/1707.06347",
    "gae": "https://arxiv.org/abs/1506.02438",
    "gym_termination": "https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/",
    "pytorch_attention": "https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention",
    "kv_cache": "https://huggingface.co/docs/transformers/kv_cache",
    "reproducibility": "https://docs.pytorch.org/docs/stable/notes/randomness.html",
}
PAD, BOS, EOS, IGNORE = 0, 1, 2, -100

class LessonError(ValueError):
    pass


def require(condition: bool, message: str) -> None:
    if not bool(condition):
        raise LessonError(message)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, allow_nan=False, separators=(",", ":"))


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def tensor_hash(tensors: dict[str, torch.Tensor]) -> str:
    h = hashlib.sha256()
    for key, val in sorted(tensors.items()):
        t = val.detach().cpu().contiguous()
        h.update(canonical([key, list(t.shape), str(t.dtype)]).encode())
        h.update(bytes(t.reshape(-1).view(torch.uint8).tolist()))
    return h.hexdigest()


def save_json(path: Path, value: Any) -> None:
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True,
                               indent=2, allow_nan=False) + "\n", encoding="utf-8")


def save_jsonl(path: Path, records: list[dict]) -> None:
    path.write_text("".join(canonical(r) + "\n" for r in records), encoding="utf-8")


def seed_all(seed: int) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def resolve_device(name: str) -> torch.device:
    require(name in {"cpu", "cuda", "auto"}, "设备只能是cpu/cuda/auto")
    if name == "auto":
        name = "cuda" if torch.cuda.is_available() else "cpu"
    require(name != "cuda" or torch.cuda.is_available(), "CUDA不可用；不会静默切回CPU")
    return torch.device(name)

@dataclass(frozen=True)
class Config:
    seed: int = 6601
    dim: int = 32
    heads: int = 4
    layers: int = 2
    ff_dim: int = 64
    max_context: int = 32
    sft_steps: int = 32
    sft_lr: float = 0.003
    updates: int = 18
    prompts_per_update: int = 6
    group_size: int = 4
    inner_epochs: int = 2
    max_completion: int = 3
    lora_rank: int = 4
    lora_alpha: float = 8.0
    rl_lr: float = 0.002
    clip_epsilon: float = 0.2
    ref_kl_coef: float = 0.01
    grad_clip: float = 1.0
    critic_steps: int = 55
    critic_lr: float = 0.015
    eval_samples: int = 5

    def validate(self) -> None:
        for field in ("seed", "dim", "heads", "layers", "ff_dim", "max_context", "sft_steps",
                      "updates", "prompts_per_update", "group_size", "inner_epochs", "max_completion",
                      "lora_rank", "critic_steps", "eval_samples"):
            v = getattr(self, field)
            require(type(v) is int and v >= (0 if field == "seed" else 1), field + "需要正整数")
        require(self.group_size >= 2 and self.dim % self.heads == 0, "G>=2且dim可被heads整除")
        require(1 <= self.max_completion <= 8 and 16 <= self.max_context <= 256,
                "context/completion超出教学范围")
        require(self.sft_steps <= 1000 and self.updates <= 500 and self.inner_epochs <= 8,
                "运行预算过大")
        for field in ("sft_lr", "lora_alpha", "rl_lr", "grad_clip", "critic_lr"):
            v = getattr(self, field)
            require(type(v) in (int, float) and math.isfinite(v) and v > 0, field + "需要有限正数")
        require(type(self.ref_kl_coef) in (int, float) and math.isfinite(self.ref_kl_coef)
                and self.ref_kl_coef >= 0, "ref_kl_coef需要非负有限")
        require(type(self.clip_epsilon) in (int, float) and math.isfinite(self.clip_epsilon)
                and 0 < self.clip_epsilon < 1, "clip_epsilon需要在(0,1)")

In [2]:
torch.set_num_threads(1)
RUN_FULL = True
OUTPUT_ROOT = Path('lesson_66_outputs')
DEVICE = 'cpu'
CFG = Config()
CFG.validate()
seed_all(CFG.seed)
print('Python：', platform.python_version(), 'PyTorch：', torch.__version__)
print('CUDA可用：', torch.cuda.is_available(), '本课运行设备：', DEVICE)
print('示例配置：', asdict(CFG))

Python： 3.13.5 PyTorch： 2.10.0+cpu
CUDA可用： False 本课运行设备： cpu
示例配置： {'seed': 6601, 'dim': 32, 'heads': 4, 'layers': 2, 'ff_dim': 64, 'max_context': 32, 'sft_steps': 32, 'sft_lr': 0.003, 'updates': 18, 'prompts_per_update': 6, 'group_size': 4, 'inner_epochs': 2, 'max_completion': 3, 'lora_rank': 4, 'lora_alpha': 8.0, 'rl_lr': 0.002, 'clip_epsilon': 0.2, 'ref_kl_coef': 0.01, 'grad_clip': 1.0, 'critic_steps': 55, 'critic_lr': 0.015, 'eval_samples': 5}


## 1. 语言模型的状态和动作是什么？

在状态式 PPO 中，环境给出 `s_t`，Actor 执行 `a_t`。在自回归语言模型里，可对应：

- **状态**：已经给出的提示与前面生成的 token 序列；
- **动作**：在当前前缀下采样的下一枚 token；
- **轨迹**：一次生成的全部回答 token（包括 EOS）；
- **奖励**：对整份完成结果可验证的分数。

这不是说所有 LLM 训练都必然采用相同的状态定义、token 裁剪或结尾规则。特别是**截断回答没有EOS**时，奖励处理必须由任务规则提前定义；不能因模型没生成完就假装完成。GRPO的多个候选必须来自同一提示，组内对比才有意义。[DeepSeekMath](https://arxiv.org/abs/2402.03300)

本课任务：`copy:03,12=` 的正确回答是 `12` 然后 EOS。两个数都来自00–15，256个提示按固定seed划分160训练、48开发和48测试。它只测试复制与格式，不测试数学推理。回答动作集合**限制为十个数字＋EOS**，每步在此集合重新归一化，温度1，最多3枚token。

### 支持实现：固定字符词表、任务分组与规则验证器

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [3]:
class CharCodec:
    """不同于预训练分词器/ChatTemplate：固定字符表与明确回答动作集合。"""
    def __init__(self):
        self.symbols = ["<pad>", "<bos>", "<eos>"] + list("copy:0123456789,=")
        require(len(self.symbols) == len(set(self.symbols)), "词表不能重复")
        self.ids = {x: i for i, x in enumerate(self.symbols)}
        self.answer_ids = [self.ids[str(i)] for i in range(10)] + [EOS]

    def encode(self, text: str) -> list[int]:
        require(type(text) is str and all(x in self.ids for x in text), "输入含未知字符")
        return [self.ids[x] for x in text]

    def decode(self, ids: list[int]) -> str:
        require(all(type(v) is int and v in self.answer_ids and v != EOS for v in ids),
                "正文只包含数字token")
        return "".join(self.symbols[v] for v in ids)

    def contract(self) -> dict:
        return {"symbols": self.symbols, "answer_ids": self.answer_ids, "bos": BOS,
                "eos": EOS, "pad": PAD, "ignore": IGNORE,
                "generation": "restricted_categorical_temperature_1; EOS_or_fixed_horizon",
                "max_completion": 3}


def make_tasks(seed=6601) -> dict[str, list[dict]]:
    rows = [{"id": f"g{a:02d}-{b:02d}", "group": f"g{a:02d}-{b:02d}",
             "prompt": f"copy:{a:02d},{b:02d}=", "expected": f"{b:02d}"}
            for a in range(16) for b in range(16)]
    random.Random(seed).shuffle(rows)
    groups = {"train": rows[:160], "dev": rows[160:208], "test": rows[208:]}
    for split, part in groups.items():
        for row in part:
            row["split"] = split
    validate_tasks(groups)
    return groups


def validate_tasks(groups: dict) -> None:
    require(set(groups) == {"train", "dev", "test"}, "必须包含train/dev/test")
    seen = set()
    for name, rows in groups.items():
        require(type(rows) is list and bool(rows), "空数据组")
        for item in rows:
            require(item["split"] == name and item["prompt"] not in seen, "数据泄漏或错配")
            require(item["prompt"].endswith(item["expected"] + "="), "显式复制规则不匹配")
            seen.add(item["prompt"])


def score_completion(text: str, stop_reason: str, expected: str) -> dict:
    """本任务的 max_new_tokens 是真实有限回合终止：没有EOS得0。
    外部计算进程被强制中断则为未知，没有对应此函数的 stop_reason。
    """
    require(type(text) is str and type(expected) is str, "奖励输入必须是字符串")
    require(len(expected) == 2 and expected.isascii() and expected.isdigit(), "参考标签非法")
    require(stop_reason in {"eos", "horizon"}, "执行错误不能当0奖励")
    if stop_reason == "horizon":
        return {"reward": 0., "reason": "no_eos_within_task_horizon"}
    if len(text) != 2 or not text.isascii() or not text.isdigit():
        return {"reward": 0., "reason": "invalid_format"}
    return {"reward": float(text == expected),
            "reason": "correct" if text == expected else "wrong_value"}

In [4]:
codec = CharCodec()
tasks = make_tasks(CFG.seed)
print('划分：', {split:len(rows) for split,rows in tasks.items()})
print('全词表：', codec.symbols)
print('允许回答的动作：', [codec.symbols[x] for x in codec.answer_ids])
print('任务示例：', tasks['train'][0])
for text,stop in [('02','eos'), ('2','eos'), ('02','horizon'), ('99','eos')]:
    print(repr(text),stop,score_completion(text,stop,'02'))

划分： {'train': 160, 'dev': 48, 'test': 48}
全词表： ['<pad>', '<bos>', '<eos>', 'c', 'o', 'p', 'y', ':', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ',', '=']
允许回答的动作： ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '<eos>']
任务示例： {'id': 'g09-09', 'group': 'g09-09', 'prompt': 'copy:09,09=', 'expected': '09', 'split': 'train'}
'02' eos {'reward': 1.0, 'reason': 'correct'}
'2' eos {'reward': 0.0, 'reason': 'invalid_format'}
'02' horizon {'reward': 0.0, 'reason': 'no_eos_within_task_horizon'}
'99' eos {'reward': 0.0, 'reason': 'wrong_value'}


### EOS 与任务 cap，不能混为“外部中断”

本课固定三步任务：

- 回答 `0,2,EOS` → 正确时奖励1；EOS计入回答概率。
- 回答 `0,2,1` → 达到本课真实任务上限，没有EOS；即使前两位正确也奖励0。
- **外部**计算超时、进程被杀、网络返回不完整 → 不是同一种训练轨迹，不能自动分配任务奖励0。需要保留未知状态或重新处理。

部分实际GRPO方案会过滤没有正常结束的回答，避免错把超长推理截断当作完整答案。本课实现可选`truncate_filter`用于观察其影响，但默认不过滤固定任务上限产生的0奖励样本；这是一项明确的**任务策略选择**，不是所有项目的标准答案。[TRL GRPO 截断配置](https://huggingface.co/docs/trl/v0.28.0/en/grpo_trainer)

## 2. 一个真正的因果 Transformer 与 Q/V LoRA

结构是：token/位置Embedding → 两层预归一化因果块 → 词表输出头。隐藏维度32、四个注意力头；无Dropout。先做短SFT，冻结基座作为参考，再独立复制加 Q/V LoRA。

这是一份**自己的小语言模型**，不能把它叫成Qwen。没有RoPE、GQA、kv cache、FlashAttention、量化或真实多卡训练。Attention显式构建因果掩码；PAD key不参与注意力，而PAD query的输出最后会被loss mask忽略。

“冻结参考”是保留独立对象且停止参数梯度；`eval()`单独不足以冻结。

### 支持实现：因果模型、LoRA注入与独立的 ValueHead

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [5]:
class Attention(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.heads = heads
        self.q = nn.Linear(d, d, bias=False)
        self.k = nn.Linear(d, d, bias=False)
        self.v = nn.Linear(d, d, bias=False)
        self.out = nn.Linear(d, d, bias=False)

    def forward(self, x, valid):
        b, t, d = x.shape
        h = self.heads
        def proj(module):
            return module(x).reshape(b, t, h, d // h).transpose(1, 2)
        q, k, v = proj(self.q), proj(self.k), proj(self.v)
        scores = q @ k.transpose(-1, -2) / math.sqrt(d // h)
        causal = torch.ones(t, t, dtype=torch.bool, device=x.device).tril()
        allowed = causal[None, None] & valid[:, None, None, :]
        # BOS有效，故每个query至少有一个允许的key；padding query的输出不计分。
        weights = torch.softmax(scores.masked_fill(~allowed, -torch.inf), dim=-1)
        return self.out((weights @ v).transpose(1, 2).contiguous().reshape(b, t, d))


class Block(nn.Module):
    def __init__(self, d, heads, hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.attn = Attention(d, heads)
        self.norm2 = nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, hidden), nn.GELU(), nn.Linear(hidden, d))

    def forward(self, x, mask):
        x = x + self.attn(self.norm1(x), mask)
        return x + self.ff(self.norm2(x))


class TinyLM(nn.Module):
    def __init__(self, vocab, cfg: Config):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.tokens = nn.Embedding(vocab, cfg.dim)
        self.pos = nn.Embedding(cfg.max_context, cfg.dim)
        self.blocks = nn.ModuleList([Block(cfg.dim, cfg.heads, cfg.ff_dim) for _ in range(cfg.layers)])
        self.norm = nn.LayerNorm(cfg.dim)
        self.head = nn.Linear(cfg.dim, vocab, bias=False)

    def forward(self, ids: torch.Tensor, valid: torch.Tensor | None = None,
                return_hidden: bool = False):
        require(ids.ndim == 2 and ids.dtype == torch.long, "需要[B,T] long输入")
        b, t = ids.shape
        require(b > 0 and 0 < t <= self.cfg.max_context, "上下文长度错误")
        if valid is None:
            valid = ids != PAD
        require(valid.shape == ids.shape and valid.dtype == torch.bool, "attention mask非法")
        require(bool(valid[:, 0].all()) and not bool(((~valid[:, :-1]) & valid[:, 1:]).any()),
                "只支持BOS有效且右补齐的batch")
        x = self.tokens(ids) + self.pos(torch.arange(t, device=ids.device))[None]
        for block in self.blocks:
            x = block(x, valid)
        hidden = self.norm(x)
        logits = self.head(hidden)
        return (logits, hidden) if return_hidden else logits


class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int, alpha: float):
        super().__init__()
        require(type(base) is nn.Linear and type(rank) is int and rank > 0,
                "LoRA基座/rank不合法")
        self.base = base.requires_grad_(False)
        self.rank = rank
        self.alpha = float(alpha)
        self.A = nn.Parameter(base.weight.new_empty(rank, base.in_features))
        self.B = nn.Parameter(base.weight.new_zeros(base.out_features, rank))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    def forward(self, x):
        return self.base(x) + (self.alpha / self.rank) * F.linear(F.linear(x, self.A), self.B)


def with_lora(base: TinyLM, cfg: Config) -> TinyLM:
    require(not any(isinstance(x, LoRALinear) for x in base.modules()), "重复注入LoRA")
    model = copy.deepcopy(base).requires_grad_(False)
    for block in model.blocks:
        block.attn.q = LoRALinear(block.attn.q, cfg.lora_rank, cfg.lora_alpha)
        block.attn.v = LoRALinear(block.attn.v, cfg.lora_rank, cfg.lora_alpha)
    return model


def adapter_state(model: TinyLM) -> dict[str, torch.Tensor]:
    return {name: p.detach().cpu().clone() for name, p in model.named_parameters()
            if name.endswith((".A", ".B"))}


def base_hash(model: TinyLM) -> str:
    # 注入LoRA后q/v权重从attn.q.weight改名为attn.q.base.weight；规范化后可比较。
    return tensor_hash({n.replace(".base.weight", ".weight"): p
                        for n, p in model.state_dict().items()
                        if not n.endswith((".A", ".B"))})


def load_adapter(model: TinyLM, adapter: dict) -> None:
    params = dict(model.named_parameters())
    require(set(adapter) == set(adapter_state(model)), "adapter字段不匹配")
    with torch.no_grad():
        for name, val in adapter.items():
            require(isinstance(val, torch.Tensor) and params[name].shape == val.shape
                    and bool(torch.isfinite(val).all()), "adapter尺寸/数值不匹配")
            params[name].copy_(val.to(device=params[name].device, dtype=params[name].dtype))


class ValueHead(nn.Module):
    """独立价值头：只看冻结语言模型在每个token前的hidden，不与GRPO混用。"""
    def __init__(self, dim: int):
        super().__init__()
        self.linear = nn.Linear(dim, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.linear(features).squeeze(-1)

In [6]:
small_model = TinyLM(len(codec.symbols), CFG)
small_ref = copy.deepcopy(small_model).requires_grad_(False).eval()
small_policy = with_lora(small_ref, CFG)
print('基座参数：', sum(p.numel() for p in small_ref.parameters()))
print('可训练LoRA参数：', sum(p.numel() for p in small_policy.parameters() if p.requires_grad))
print('参考完全冻结：', all(not p.requires_grad for p in small_ref.parameters()))
print('LoRA初始B为零：', all(torch.count_nonzero(p) == 0 for n,p in small_policy.named_parameters() if n.endswith('.B')))
print('同一基座内容：', base_hash(small_policy) == tensor_hash(small_ref.state_dict()))

基座参数： 19200
可训练LoRA参数： 1024
参考完全冻结： True
LoRA初始B为零： True
同一基座内容： True


## 3. 标签与 token 级概率：只移位一次

本课输入是 `[BOS] + prompt字符 + 回答token`。同位标签为：

```text
输入： BOS c o p y : 0 3 , 1 2 =  1 2 EOS PAD PAD
标签： -100 ............ -100   1 2 EOS -100 -100
```

对第一个回答 token 的预测来自**它的前一个位置**的 logits。`token_values`只移位一次，不把已经对齐的 labels 再移动。整份回答概率是各 token 的条件概率乘积，log 概率是**和**。

如果真实模型的内部loss已帮你shift过标签，不能照抄本课再shift。对于真实tokenizer，还要检查ChatTemplate、EOS、回答起点、padding侧和截断边界。本课的字符 tokenizer 没有这些预训练接口。

### 支持实现：同位标签、受限动作归一化、token概率与 SFT

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [7]:
def make_batch(rows: list[dict], answers: list[list[int]], codec: CharCodec,
               cfg: Config, device: torch.device | str = "cpu") -> dict:
    require(len(rows) > 0 and len(rows) == len(answers), "样本/回答长度不匹配")
    full, labels, masks, prefix_lengths = [], [], [], []
    for row, seq in zip(rows, answers):
        require(type(seq) is list and 1 <= len(seq) <= cfg.max_completion, "完成序列长度非法")
        require(all(type(a) is int and a in codec.answer_ids for a in seq), "回答动作不在词表")
        require(EOS not in seq[:-1], "EOS后有多余token")
        prefix = [BOS] + codec.encode(row["prompt"])
        combined = prefix + seq
        require(len(combined) <= cfg.max_context, "样本超上下文，不得静默截断")
        full.append(combined)
        labels.append([IGNORE] * len(prefix) + seq)
        masks.append([False] * len(prefix) + [True] * len(seq))
        prefix_lengths.append(len(prefix))
    width = max(map(len, full))
    ids = torch.tensor([seq + [PAD] * (width-len(seq)) for seq in full],
                       device=device, dtype=torch.long)
    targets = torch.tensor([seq + [IGNORE] * (width-len(seq)) for seq in labels],
                           device=device, dtype=torch.long)
    completion_mask = torch.tensor([seq + [False] * (width-len(seq)) for seq in masks],
                                   device=device, dtype=torch.bool)
    attention = ids != PAD
    require(torch.equal(completion_mask, targets != IGNORE), "label/mask不一致")
    return {"ids": ids, "labels": targets, "completion_mask": completion_mask,
            "attention": attention, "prefix_lengths": prefix_lengths}


def action_distribution(logits: torch.Tensor, codec: CharCodec) -> torch.Tensor:
    """完整token输出中只对digit+EOS动作重新归一化。"""
    require(logits.is_floating_point() and bool(torch.isfinite(logits).all()), "logits必须有限")
    require(logits.shape[-1] == len(codec.symbols), "词表尺寸不符")
    return F.log_softmax(logits[..., codec.answer_ids], dim=-1)


def token_values(logits: torch.Tensor, batch: dict,
                 codec: CharCodec) -> dict[str, torch.Tensor]:
    """输入labels同位；唯一一次移位。含 EOS，排除prompt与PAD。"""
    labels = batch["labels"]
    require(labels.dtype == torch.long and labels.shape == logits.shape[:2], "标签尺寸/type错误")
    require(bool((labels[:, 0] == IGNORE).all()), "首位置必须忽略")
    target = labels[:, 1:]
    mask = batch["completion_mask"][:, 1:]
    require(torch.equal(mask, target != IGNORE) and bool(mask.any(-1).all()), "监督mask无效")
    mapper = labels.new_full((len(codec.symbols),), -1)
    mapper[codec.answer_ids] = torch.arange(len(codec.answer_ids), device=labels.device)
    safe = target.masked_fill(~mask, EOS)
    require(bool(((safe >= 0) & (safe < len(codec.symbols))).all()), "标签越界")
    action_idx = mapper[safe]
    require(bool((action_idx >= 0).all()), "标签不是允许的动作")
    all_lp = action_distribution(logits[:, :-1], codec)
    selected = all_lp.gather(-1, action_idx.unsqueeze(-1)).squeeze(-1)
    return {"token_logps": selected.masked_fill(~mask, 0),
            "token_distribution": all_lp, "mask": mask,
            "counts": mask.sum(-1), "sequence_logps": selected.masked_fill(~mask, 0).sum(-1)}


def warmup_sft(model: TinyLM, rows: list[dict], codec: CharCodec, cfg: Config) -> list[dict]:
    require(all(r["split"] == "train" for r in rows), "SFT不得用测试样本")
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.sft_lr)
    rng = random.Random(cfg.seed+11)
    device = next(model.parameters()).device
    history = []
    model.train()
    for step in range(cfg.sft_steps):
        sample = rng.sample(rows, min(32, len(rows)))
        seq = [codec.encode(r["expected"]) + [EOS] for r in sample]
        batch = make_batch(sample, seq, codec, cfg, device)
        lp = token_values(model(batch["ids"], batch["attention"]), batch, codec)
        loss = -lp["token_logps"].sum() / lp["counts"].sum()
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
        optimizer.step()
        if step == 0 or (step+1) % 8 == 0 or step+1 == cfg.sft_steps:
            history.append({"step": step+1, "token_nll": float(loss.detach())})
    model.eval()
    for p in model.parameters(): p.grad = None
    return history

def extract_completion_positions(token_data: dict, cfg: Config) -> dict:
    """把整条输入的监督位置收集成[N,L]；全程不改变时间顺序与梯度。
    原prompt位置被ignore，但完整forward仍让回答attend到prompt。
    """
    lp=token_data["token_logps"]
    distributions=token_data["token_distribution"]
    valid=token_data["mask"]
    require(lp.ndim==2 and distributions.shape[:2]==lp.shape and valid.shape==lp.shape,
            "token收集器输入形状错误")
    require(bool((valid.sum(-1)>0).all()) and bool((valid.sum(-1)<=cfg.max_completion).all()),
            "每行有效目标数异常")
    selected=[];all_dist=[];packed_masks=[]
    for i in range(lp.shape[0]):
        count=int(valid[i].sum())
        selected.append(F.pad(lp[i,valid[i]],(0,cfg.max_completion-count)))
        all_dist.append(F.pad(distributions[i,valid[i]],
                              (0,0,0,cfg.max_completion-count),
                              value=-math.log(distributions.shape[-1])))
        packed_masks.append(torch.arange(cfg.max_completion,device=lp.device)<count)
    return {"token_logps":torch.stack(selected),"token_distribution":torch.stack(all_dist),
            "mask":torch.stack(packed_masks),"counts":valid.sum(-1)}

In [8]:
demo_tasks = tasks['train'][:2]
demo_answers = [codec.encode(demo_tasks[0]['expected'])+[EOS], [EOS]]
demo_batch = make_batch(demo_tasks, demo_answers, codec, CFG)
logits = torch.zeros(*demo_batch['ids'].shape, len(codec.symbols), dtype=torch.float64, requires_grad=True)
original = token_values(logits,demo_batch,codec)
print('原始监督目标数：',original['counts'].tolist())
print('受限11动作、有效回答的log概率和：',original['sequence_logps'].detach().tolist())
print('EOS个数：',int((demo_batch['labels']==EOS).sum()))
print('提示仍是有效上下文：',bool(demo_batch['attention'][:,0].all()))

原始监督目标数： [3, 1]
受限11动作、有效回答的log概率和： [-7.193685818395112, -2.3978952727983707]
EOS个数： 2
提示仍是有效上下文： True


### 将整条提示的监督位置收集到一个小矩阵

模型前向得到 `[N,T,V]`，其中大部分位置是提示。训练GRPO只需要回答位置的每-token值。本课**不按字符串查找答案起点**，而是根据布尔mask把它们依次收集成 `[N,max_completion]`。

无效位置的 `token_logps` 置零，完整动作分布填成正规化的均匀分布，以便独立校验KL；但训练时依然用布尔mask把它们完全排除。

这项转换必须保留梯度，不能为了方便保存为 Python 浮点列表再训练。

In [9]:
collected = extract_completion_positions(original,CFG)
print('有效token数：',collected['counts'].tolist())
print('收集后的token log概率：',collected['token_logps'].detach().tolist())
print('收集后的mask：',collected['mask'].tolist())
print('EOS后的无效位置已填零，不会给token目标打分。')

有效token数： [3, 1]
收集后的token log概率： [[-2.3978952727983707, -2.3978952727983707, -2.3978952727983707], [-2.3978952727983707, 0.0, 0.0]]
收集后的mask： [[True, True, True], [True, False, False]]
EOS后的无效位置已填零，不会给token目标打分。


## 4. 组奖励：同一道题 G 条独立回答

设同题组内完成级奖励为 $R_1,\ldots,R_G$。最简单的组相对优势为：

\[
A_i=R_i-\overline R
\]

若按照组内总体标准差缩放，则为：

\[
A_i=\frac{R_i-\overline R}{\operatorname{std}_{population}(R)+\epsilon}
\]

与价值网络的 $V(s)$ 不同，它不估计长期状态价值，而只比较当前题目采样组中的结果。**同组全同分时，优势全零**：它可能意味着都正确，也可能意味着都错误；不能从零优势判断模型能力。

训练器会在每次更新前固定组优势，然后在这批回答的多次参数更新中保持它不变。这避免把变动分数当成固定旧数据复用。

### 支持实现：逐token实际采样、组内奖励、旧概率封存

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [10]:
@torch.no_grad()
def rollout(model: TinyLM, prompts: list[dict], codec: CharCodec,
            cfg: Config, generator: torch.Generator, group_size: int | None = None,
            greedy: bool = False) -> dict:
    """模型实际生成，绝不注入正确答案。每步采样后出现EOS的行不再产生动作。"""
    group = cfg.group_size if group_size is None else group_size
    require(bool(prompts) and type(group) is int and group >= 1, "问题列表/组大小非法")
    expanded = [copy.deepcopy(p) for p in prompts for _ in range(group)]
    require(all(len(p["prompt"]) == len(expanded[0]["prompt"]) for p in expanded),
            "本课简化采样器要求提示等长，不做左补齐")
    prefix = [[BOS] + codec.encode(row["prompt"]) for row in expanded]
    require(len(prefix[0]) + cfg.max_completion <= cfg.max_context, "生成视野超上下文")
    device = next(model.parameters()).device
    ids = torch.tensor(prefix, dtype=torch.long, device=device)
    live = torch.ones(len(expanded), dtype=torch.bool, device=device)
    answers = [[] for _ in expanded]
    sampled_logps = [[] for _ in expanded]
    was_training = model.training
    model.eval()
    try:
        for _ in range(cfg.max_completion):
            logits = model(ids, ids != PAD)[:, -1]
            logp = action_distribution(logits, codec)
            choice = logp.argmax(-1) if greedy else torch.multinomial(logp.exp(), 1, generator=generator).squeeze(-1)
            token = torch.tensor(codec.answer_ids, device=device)[choice]
            selected = logp.gather(1, choice[:, None]).squeeze(-1)
            old_live = live.clone()
            for i in range(len(expanded)):
                if bool(old_live[i]):
                    answers[i].append(int(token[i]))
                    sampled_logps[i].append(float(selected[i]))
            live &= token != EOS
            appended = token.masked_fill(~old_live, PAD)
            ids = torch.cat([ids, appended[:, None]], dim=1)
            if not bool(live.any()):
                break
    finally:
        model.train(was_training)
    samples = []
    for i, (row, ans) in enumerate(zip(expanded, answers)):
        terminated = ans[-1] == EOS
        body = ans[:-1] if terminated else ans
        text = codec.decode(body)
        reason = "eos" if terminated else "horizon"
        score = score_completion(text, reason, row["expected"])
        samples.append({"sample_id": f"{row['id']}-s{i%group}", "task_id": row["id"],
                        "group_index": i//group, "slot": i%group,
                        "token_ids": ans, "text": text, "stop_reason": reason,
                        "sampled_logps": sampled_logps[i], **score})
    return {"tasks": expanded, "answers": answers, "samples": samples,
            "group_size": group, "prompt_count": len(prompts),
            "policy_hash": tensor_hash(model.state_dict()),
            "codec_hash": digest(codec.contract()), "greedy": greedy}


def group_advantages(rewards: torch.Tensor, scaling: str = "group",
                     eps: float = 1e-6) -> torch.Tensor:
    require(rewards.ndim == 2 and rewards.shape[1] >= 2 and rewards.is_floating_point(),
            "奖励需要[B,G], G>=2")
    require(bool(torch.isfinite(rewards).all()), "奖励不能包含非有限值")
    require(scaling in {"group", "none"}, "scaling只能是group/none")
    require(type(eps) in (float, int) and math.isfinite(eps) and eps > 0,
            "eps必须有限正数")
    frozen = rewards.detach()
    centered = frozen - frozen.mean(1, keepdim=True)
    if scaling == "group":
        return centered / (frozen.std(1, unbiased=False, keepdim=True) + eps)
    return centered


def align_rollout(roll: dict, model: TinyLM, reference: TinyLM,
                  codec: CharCodec, cfg: Config, scaling: str) -> dict:
    """采样后、更新前封存 old token probabilities、reference categorical分布和组优势。"""
    require(not roll["greedy"], "贪心输出不能冒充随机轨迹")
    require(roll["policy_hash"] == tensor_hash(model.state_dict()), "重算前策略已改变")
    require(roll["codec_hash"] == digest(codec.contract()), "回答动作集合发生变化")
    require(all(r["split"] == "train" for r in roll["tasks"]), "训练只接收train")
    batch = make_batch(roll["tasks"], roll["answers"], codec, cfg, next(model.parameters()).device)
    with torch.no_grad():
        now = extract_completion_positions(token_values(model(batch["ids"], batch["attention"]), batch, codec),cfg)
        ref = extract_completion_positions(token_values(reference(batch["ids"], batch["attention"]), batch, codec),cfg)
    old = now["token_logps"]
    for i, sample in enumerate(roll["samples"]):
        real = old[i][now["mask"][i]].tolist()
        require(len(real) == len(sample["sampled_logps"])
                and max(abs(a-b) for a,b in zip(real,sample["sampled_logps"])) < 3e-5,
                "逐步采样概率与teacher forcing重算不一致")
        require(sample["token_ids"] == roll["answers"][i], "回答token与记录不一致")
    rewards = old.new_tensor([sample["reward"] for sample in roll["samples"]])
    b,g = roll["prompt_count"],roll["group_size"]
    require(len(rewards) == b*g, "组大小错误")
    adv = group_advantages(rewards.reshape(b,g),scaling).reshape(-1)
    return {"batch": batch, "old_logps": old.detach().clone(),
            "reference_distribution": ref["token_distribution"].detach().clone(),
            "mask": now["mask"].detach().clone(), "advantages": adv.detach().clone(),
            "rewards": rewards.detach().clone(), "samples": copy.deepcopy(roll["samples"]),
            "policy_hash": roll["policy_hash"], "updated": 0,
            "frozen_signature": digest({"samples": roll["samples"], "codec": codec.contract(),
                                         "group": g, "scaling":scaling})}

In [11]:
reward_groups = torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]],dtype=torch.float64)
print('只中心化：',group_advantages(reward_groups,'none').tolist())
print('按组std缩放：',group_advantages(reward_groups,'group').tolist())
print('全同分的组：',group_advantages(reward_groups,'group')[1].tolist())

只中心化： [[0.75, -0.25, -0.25, -0.25], [0.0, 0.0, 0.0, 0.0]]
按组std缩放： [[1.7320468075781148, -0.5773489358593716, -0.5773489358593716, -0.5773489358593716], [0.0, 0.0, 0.0, 0.0]]
全同分的组： [0.0, 0.0, 0.0, 0.0]


## 5. 先真实采样，再与 teacher forcing 核对

采样过程中有EOS的行立即停止；其他行继续，已经结束的行只接PAD。每条完成记录保留 token IDs、停止原因、规则奖励和当时的逐token对数概率。

随后在参数还未更新前重新前向，**采样时记录的token logp应当与teacher forcing重新计算的值一致**。这可以发现温度、Top-p、动作集合、移位或Dropout两边不匹配。

组内回答可能重复；不能为了让奖励“更丰富”而强行去重或仅保留成功样本，否则改变了实际采样分布。

In [12]:
torch.set_num_threads(1)
seed_all(6602)
example_roll = rollout(small_policy,tasks['train'][:2],codec,CFG,
                       torch.Generator().manual_seed(66),group_size=4)
for x in example_roll['samples']:
    print(x['task_id'], '槽',x['slot'],repr(x['text']),x['stop_reason'],'reward=',x['reward'])
example_fixed=align_rollout(example_roll,small_policy,small_ref,codec,CFG,'group')
print('逐步采样与重算一致；已封存的token形状：',tuple(example_fixed['old_logps'].shape))
print('组优势：',example_fixed['advantages'].reshape(2,4).tolist())

g09-09 槽 0 '742' horizon reward= 0.0
g09-09 槽 1 '610' horizon reward= 0.0
g09-09 槽 2 '781' horizon reward= 0.0
g09-09 槽 3 '719' horizon reward= 0.0
g05-10 槽 0 '089' horizon reward= 0.0
g05-10 槽 1 '672' horizon reward= 0.0
g05-10 槽 2 '7' eos reward= 0.0
g05-10 槽 3 '630' horizon reward= 0.0
逐步采样与重算一致；已封存的token形状： (8, 3)
组优势： [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]


## 6. 三种 token loss 的分母不是代码小优化

令每个有效token的裁剪代理项为 $l_{i,t}$：

\[
l_{i,t}=\min\big(r_{i,t}A_i,\operatorname{clip}(r_{i,t},1-\epsilon,1+\epsilon)A_i\big)
\]

这里 $r_{i,t}=\pi_\theta(o_{i,t}|s_{i,t})/\pi_{old}(o_{i,t}|s_{i,t})$ 是**单token概率比**，不是上课的完整序列比。

按当前官方TRL文档的公式形式，可辨认三种分母：[GRPO Trainer v0.28.0](https://huggingface.co/docs/trl/v0.28.0/en/grpo_trainer)

- **GRPO型**：每条回答先除以其有效长度，再按回答数平均。
- **DAPO型**：所有有效token代理项求和，除以整个批次有效token数。
- **Dr. GRPO型**：全批次求和，除以固定 `回答数×最大回答长度`。

不同分母改变每条token、每个回答对梯度的权重，不是同一种loss写法的等价加速。第三种使用固定常量，是为了研究长度偏差。实际论文/框架还包含更多约定，本课明确把参考KL另作一个独立项。

### 支持实现：token-level PPO裁剪、三种归约和访问前缀上的参考KL

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [13]:
def token_clip_loss(new_logps: torch.Tensor, old_logps: torch.Tensor,
                    mask: torch.Tensor, advantage: torch.Tensor, epsilon: float,
                    variant: str, max_completion: int,
                    truncate_mask: torch.Tensor | None = None) -> tuple[torch.Tensor, dict]:
    """等权组样本；GRPO按每条有效长度、DAPO按全batch有效token、Dr按全局常量归约。
    策略目标是token-level importance ratio + clip，优势为completion-level常数。
    """
    require(new_logps.ndim == 2 and old_logps.shape == new_logps.shape,
            "token log概率需要[N,T]")
    require(mask.shape == new_logps.shape and mask.dtype == torch.bool,
            "token mask形状/type错误")
    require(advantage.ndim == 1 and advantage.shape[0] == new_logps.shape[0],
            "completion优势需要[N]")
    require(all(bool(torch.isfinite(t).all()) for t in (new_logps,old_logps,advantage)),
            "概率或优势非有限")
    require(variant in {"grpo", "dapo", "dr_grpo"}, "损失类型不支持")
    require(type(max_completion) is int and max_completion > 0 and new_logps.shape[1] <= max_completion,
            "最大长度不符合任务契约")
    require(type(epsilon) in (int,float) and math.isfinite(epsilon) and 0 < epsilon < 1,
            "epsilon必须在(0,1)")
    effective = mask.clone()
    if truncate_mask is not None:
        require(truncate_mask.shape == advantage.shape and truncate_mask.dtype == torch.bool,
                "截断筛选mask应为[N] bool")
        effective &= ~truncate_mask[:,None]
    if not bool(effective.any()):
        # 全部过滤：没有要优化的token。返回零图以便安全地观察，调用方必须跳过optimizer.step。
        zero = new_logps.sum()*0
        return zero, {"active_tokens":0, "eligible_completions":0,
                      "skipped":True, "clip_fraction":None}
    log_ratio = new_logps - old_logps.detach()
    # 无效位置用0，避免不必要的exp溢出；有效位置极端漂移则拒绝而非静默修正。
    safe = log_ratio.masked_fill(~effective, 0.)
    require(bool((safe.detach().abs() < 40).all()), "比值变化太大，停止并诊断")
    ratio = safe.exp()
    a = advantage.detach()[:,None]
    plain = ratio*a
    clipped = ratio.clamp(1-epsilon,1+epsilon)*a
    obj = torch.minimum(plain,clipped).masked_fill(~effective,0.)
    counts = effective.sum(1)
    if variant == "grpo":
        # 被过滤的completion贡献零，但计划槽仍留在均值分母。
        loss = -(obj.sum(1) / counts.clamp_min(1)).mean()
    elif variant == "dapo":
        loss = -obj.sum() / effective.sum()
    else:
        loss = -obj.sum() / (new_logps.shape[0] * max_completion)
    active = (((ratio > 1+epsilon)&(a > 0))|((ratio < 1-epsilon)&(a < 0))) & effective
    return loss, {"active_tokens":int(effective.sum()),
                  "eligible_completions":int((counts>0).sum()),"skipped":False,
                  "clip_fraction":float(active.sum()/effective.sum()),
                  "unclipped_mean_ratio":float(ratio[effective].detach().mean())}


def visited_categorical_kl(current: torch.Tensor, reference: torch.Tensor,
                           mask: torch.Tensor) -> torch.Tensor:
    """在旧轨迹访问的前缀计算完整回答动作分布KL(current||reference)。
    每个前缀处精确；状态的加权仍受旧rollout访问分布限制，不是全模型KL。
    """
    require(current.shape == reference.shape and current.ndim == 3,
            "KL分布需要[N,T,A]")
    require(mask.shape == current.shape[:-1] and mask.dtype == torch.bool and bool(mask.any()),
            "KL mask需要有效token")
    require(bool(torch.isfinite(current).all()) and bool(torch.isfinite(reference).all()),
            "KL概率非有限")
    for value in (current, reference):
        require(torch.allclose(value.logsumexp(-1),torch.zeros_like(value[...,0]),
                               rtol=2e-5,atol=2e-5),"输入应为已归一化的log分布")
    kl = (current.exp() * (current - reference.detach())).sum(-1)
    return kl[mask].mean()


def grpo_update(policy: TinyLM, reference: TinyLM, fixed: dict, codec: CharCodec,
                cfg: Config, optimizer, variant: str, truncate_filter=False) -> dict:
    require(type(truncate_filter) is bool, "truncate_filter需要bool")
    require(all(not p.requires_grad for p in reference.parameters()), "参考必须冻结")
    trainable = [p for p in policy.parameters() if p.requires_grad]
    require({id(p) for group in optimizer.param_groups for p in group["params"]} ==
            {id(p) for p in trainable}, "优化器必须只包含当前LoRA参数")
    if fixed["updated"] == 0:
        require(fixed["policy_hash"] == tensor_hash(policy.state_dict()), "旧数据采样后策略被改动")
    batch = fixed["batch"]
    mask = fixed["mask"]
    truncation = None
    if truncate_filter:
        truncation = torch.tensor([s["stop_reason"] == "horizon" for s in fixed["samples"]],
                                  dtype=torch.bool,device=mask.device)
    optimizer.zero_grad(set_to_none=True)
    logits = policy(batch["ids"],batch["attention"])
    out = extract_completion_positions(token_values(logits,batch,codec),cfg)
    loss,details = token_clip_loss(out["token_logps"],fixed["old_logps"],mask,
                                   fixed["advantages"],cfg.clip_epsilon,variant,
                                   cfg.max_completion,truncation)
    if details["skipped"]:
        return {"updated":False,"reason":"no_eligible_tokens","policy_loss":0.,
                "kl":None,**details}
    used_mask = mask if truncation is None else mask & ~truncation[:,None]
    kl = visited_categorical_kl(out["token_distribution"],fixed["reference_distribution"],used_mask)
    total = loss + cfg.ref_kl_coef*kl
    total.backward()
    require(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in trainable),
            "LoRA梯度缺失或非有限")
    require(all(p.grad is None for p in reference.parameters()), "参考不应收到梯度")
    norm = nn.utils.clip_grad_norm_(trainable,cfg.grad_clip,error_if_nonfinite=True)
    optimizer.step()
    fixed["updated"] += 1
    return {"updated":True,"policy_loss":float(loss.detach()),
            "visited_kl":float(kl.detach()),"total_loss":float(total.detach()),
            "grad_norm_before_clip":float(norm),"inner_step":fixed["updated"],**details}

In [14]:
new=torch.zeros(2,3,dtype=torch.float64,requires_grad=True)
old=torch.zeros_like(new)
valid=torch.tensor([[True,False,False],[True,True,True]])
adv=torch.tensor([1.,-1.])
for kind in ['grpo','dapo','dr_grpo']:
    loss,info=token_clip_loss(new,old,valid,adv,.2,kind,3)
    print(kind,'loss=',float(loss.detach()),'active_tokens=',info['active_tokens'])
print('只有长度不同，三种归约就不一定相同。')

grpo loss= -0.0 active_tokens= 4
dapo loss= 0.5 active_tokens= 4
dr_grpo loss= 0.3333333333333333 active_tokens= 4
只有长度不同，三种归约就不一定相同。


### 正负优势和裁剪仍要分方向

设正优势、ratio=1.6，超过 `1+epsilon=1.2`：该项处于裁剪平坦区。负优势、ratio=0.5，低于 `1-epsilon=0.8`：也进入平坦区。方向反过来则仍保留纠正梯度。

本课不使用 `clamp(ratio)*A` 代替 `min`。即使某项被裁剪，也不代表实际参数更新后的ratio绝不会越界——这是目标函数裁剪，不是硬投影。[PPO 原论文](https://arxiv.org/abs/1707.06347)

此外，可选择把没有EOS的完成完全排除出策略loss：只将它们mask为0，但**计划样本与日志仍然存在**。若全部被排除，必须跳过optimizer更新；不能把零token数量当分母，也不能伪造一次有效训练。

In [15]:
boundary_new=torch.tensor([[math.log(1.6),0.,0.],[math.log(.5),0.,0.]],
                          dtype=torch.float64,requires_grad=True)
mask=torch.tensor([[True,False,False],[True,False,False]])
loss,summary=token_clip_loss(boundary_new,torch.zeros_like(boundary_new),mask,
                             torch.tensor([1.,-1.]),.2,'grpo',3)
loss.backward()
print('两种已经裁剪的token梯度：',boundary_new.grad[:,0].tolist())
filtered,info=token_clip_loss(new,old,valid,adv,.2,'grpo',3,
                              torch.tensor([True,True]))
print('全部回答被过滤：',info,'loss=',float(filtered.detach()))

两种已经裁剪的token梯度： [0.0, 0.0]
全部回答被过滤： {'active_tokens': 0, 'eligible_completions': 0, 'skipped': True, 'clip_fraction': None} loss= 0.0


## 7. 固定参考KL：区分逐token完整分布与采样单个动作

对已访问的回答前缀，我们有当前模型与参考模型在11个合法动作上的完整分布，因此可精确计算该前缀的：

\[
D_{KL}(\pi_\theta(\cdot|s_t)\|\pi_{ref}(\cdot|s_t))
\]

然后只在有效回答位置上平均，作为额外惩罚。这是**被旧轨迹访问到的状态上的分类KL**；没有枚举所有可能未来前缀，不是完整语言策略的精确全局KL。

它与GRPO文献中常见的基于采样动作的KL估计并非同一实现。这里利用动作空间只有11个的便利，核对梯度与符号。输入log分布必须先正规化；不能把负的单样本log比值一律截成零。

本课四条训练分支都使用同一访问状态KL系数0.01，以便比较token归约；它不是训练结果安全性或事实正确性的保证。

In [16]:
uniform=torch.full((1,2,11),-math.log(11),dtype=torch.float64)
mask=torch.tensor([[True,False]])
print('同分布KL：',float(visited_categorical_kl(uniform,uniform,mask)))
shifted=uniform.clone();shifted[0,0,0]+=1
shifted[0,0]=F.log_softmax(shifted[0,0],-1)
print('参考有所变化时的KL：',float(visited_categorical_kl(uniform,shifted,mask)))

同分布KL： 0.0


参考有所变化时的KL： 0.05423610870187312


## 8. 价值函数是不是“组内平均奖励”？

不是。价值函数估计**给定当前前缀，未来可能取得的回报**，即状态的条件期望；组相对优势只使用当次同题 G 条回答的相对表现。它们都是 baseline 思想的实现，但训练信息与用途不同。

本课让 SFT 语言模型保持冻结，抽取每个回答token之前的隐藏状态，训练一个独立的**线性 ValueHead**预测该token起至结束的Monte-Carlo折扣回报。它不参与四条GRPO actor更新，这样可以观察 value loss 而不把critic悄悄混进GRPO。

训练/开发数据来自不同提示，评估误差不用于挑选某轮actor。奖励很稀疏、任务简单、值头很小，因此误差下降不能证明价值模型适合长推理链，也不能自动推出PPO比组方法更强。

### 支持实现：终止、外部截断的GAE；冻结语言模型的独立价值头

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [17]:
def gae_with_boundaries(reward: torch.Tensor, value: torch.Tensor,
                        next_value: torch.Tensor, mask: torch.Tensor,
                        terminated: torch.Tensor, truncated: torch.Tensor,
                        gamma: float = 0.95, lam: float = 0.95) -> dict:
    """每条轨迹为[B,T]，末个有效位置必须terminated或external truncated。

    terminated：真实状态结束，不对next_value bootstrap。
    truncated：外部记录被截断，但另提供可验证的next_value；这一行的delta bootstrap，
               GAE递推在该位置停止（下一动作未被采集）。
    本课三步回答cap是任务定义的terminal，不属于external truncated。
    """
    require(all(isinstance(x, torch.Tensor) and x.shape == reward.shape for x in
                (value,next_value,mask,terminated,truncated)), "GAE张量形状不一致")
    require(reward.ndim == 2 and reward.is_floating_point() and mask.dtype == torch.bool
            and terminated.dtype == torch.bool and truncated.dtype == torch.bool,
            "GAE需要[B,T]奖励与bool mask/done")
    require(all(bool(torch.isfinite(x).all()) for x in (reward,value,next_value)),
            "GAE输入非有限")
    require(type(gamma) in (float,int) and math.isfinite(gamma) and 0 <= gamma <= 1
            and type(lam) in (float,int) and math.isfinite(lam) and 0 <= lam <= 1,
            "gamma/lambda需要在[0,1]")
    require(bool(mask.any(1).all()), "每条轨迹至少一个有效动作")
    require(not bool(((~mask[:,:-1]) & mask[:,1:]).any()), "GAE不接受中间mask空洞")
    require(not bool((terminated & truncated).any()), "同一位置不能同时终止和截断")
    end = terminated | truncated
    lengths = mask.sum(1)
    for i in range(mask.shape[0]):
        last = int(lengths[i])-1
        require(bool(end[i,last]) and not bool(end[i,:last].any())
                and not bool(end[i,last+1:].any()), "每条轨迹只在末个有效位置终止或截断")
    rewards, values, next_values = reward.detach(),value.detach(),next_value.detach()
    delta = rewards + gamma*(~terminated).to(reward.dtype)*next_values - values
    delta = delta.masked_fill(~mask,0)
    advantages = torch.zeros_like(reward)
    carry = torch.zeros(reward.shape[0],dtype=reward.dtype,device=reward.device)
    for t in reversed(range(reward.shape[1])):
        continue_episode = (mask[:,t] & ~end[:,t]).to(reward.dtype)
        carry = delta[:,t] + gamma*lam*continue_episode*carry
        carry = carry*mask[:,t].to(reward.dtype)
        advantages[:,t] = carry
    return {"advantage":advantages,"return_target":advantages+values*mask,
            "delta":delta,"lengths":lengths}


def critic_features(model: TinyLM, rows: list[dict], answers: list[list[int]],
                    codec: CharCodec, cfg: Config) -> torch.Tensor:
    """冻结actor隐藏状态，抽取采样每个token前的值特征；不训练骨干。"""
    batch=make_batch(rows,answers,codec,cfg,next(model.parameters()).device)
    with torch.no_grad():
        _, hidden = model(batch["ids"],batch["attention"],return_hidden=True)
    features=[]
    for i, seq in enumerate(answers):
        start=batch["prefix_lengths"][i]-1
        features.extend([hidden[i,start+t].detach().cpu().clone() for t in range(len(seq))])
    return torch.stack(features)


def critic_dataset(model: TinyLM, rows: list[dict], codec: CharCodec, cfg: Config,
                   seed: int, group: int = 3) -> tuple[torch.Tensor,torch.Tensor,dict]:
    roll = rollout(model, rows,codec,cfg,torch.Generator(device=next(model.parameters()).device).manual_seed(seed),group)
    features=critic_features(model,roll["tasks"],roll["answers"],codec,cfg)
    # 本课只有完成末端奖励；任务max_length也定义为terminal，故MC return无bootstrap。
    values=[]
    for seq,record in zip(roll["answers"],roll["samples"]):
        n=len(seq)
        values.extend([(.95**(n-1-i))*record["reward"] for i in range(n)])
    labels=torch.tensor(values,dtype=features.dtype)
    return features,labels,{"prompts":len(rows),"completions":len(roll["answers"]),
                             "active_tokens":len(values),"positive_rewards":int(sum(s["reward"] for s in roll["samples"]))}


def train_critic(reference: TinyLM, tasks: dict,codec: CharCodec,cfg: Config) -> tuple[dict,dict]:
    """训练独立的小价值头；不会把critic预测回传进GRPO actor更新。"""
    with torch.no_grad():
        x,y,train_cost=critic_dataset(reference,tasks["train"][:48],codec,cfg,seed=cfg.seed+401)
        xd,yd,dev_cost=critic_dataset(reference,tasks["dev"][:24],codec,cfg,seed=cfg.seed+402)
    critic=ValueHead(cfg.dim).to(dtype=x.dtype)
    opt=torch.optim.Adam(critic.parameters(),lr=cfg.critic_lr)
    before={"train_mse":float(F.mse_loss(critic(x),y).detach()),
            "dev_mse":float(F.mse_loss(critic(xd),yd).detach())}
    history=[]
    for step in range(cfg.critic_steps):
        pred=critic(x)
        loss=F.mse_loss(pred,y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if step == 0 or (step+1)%10==0 or step+1==cfg.critic_steps:
            with torch.no_grad():
                history.append({"step":step+1,"train_mse":float(F.mse_loss(critic(x),y)),
                                "dev_mse":float(F.mse_loss(critic(xd),yd))})
    after=history[-1]
    return ({"before":before,"after":after,"history":history,
             "data":{"train":train_cost,"dev":dev_cost},
             "critic_parameters":sum(p.numel() for p in critic.parameters()),
             "scope":"frozen_sft_hidden_to_MC_return; no_GAE_for_actor; no_test_selection"},
            {k:v.detach().cpu().clone() for k,v in critic.state_dict().items()})

In [18]:
critic_demo = ValueHead(CFG.dim)
print('独立ValueHead参数数：',sum(p.numel() for p in critic_demo.parameters()))
print('GRPO policy不使用critic参数。')

独立ValueHead参数数： 33
GRPO policy不使用critic参数。


## 9. 终局和外部截断：GAE最后一项必须看清

在逐动作值模型中：

\[
\delta_t=r_t+\gamma(1-\mathrm{terminated}_t)V(s_{t+1})-V(s_t)
\]
\[
A_t=\delta_t+\gamma\lambda(1-\mathrm{episodeBoundary}_t)A_{t+1}
\]

真实终止时不使用下一个状态的价值；**外部截断**且后继状态确实可评价时，最后一个delta应保留bootstrap，但不能递推到未采集的下一动作。[GAE](https://arxiv.org/abs/1506.02438) · [Gymnasium time limit](https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/)

教学检查：两步值 `[0.2,0.3]`。真实最终奖励1，γ=0.95、λ=1时return为 `[0.95,1]`。若只是外部截断、最后有可靠的 `V(next)=0.8`，最后一步target为0.76，不应该被当成0。

**重要区别**：本课生成任务的三token cap 是任务定义的终局，未生成EOS则按奖励规则为0；不是可无限继续生成、只因机器时间到了而中断的外部截断。无法验证的外部中断不能冒充有可靠bootstrap的正常样本。

In [19]:
r=torch.tensor([[0.,1.]],dtype=torch.float64)
v=torch.tensor([[.2,.3]],dtype=torch.float64)
nv=torch.tensor([[.3,0.]],dtype=torch.float64)
valid=torch.tensor([[True,True]])
terminal=torch.tensor([[False,True]])
zeros=torch.zeros_like(terminal)
a=gae_with_boundaries(r,v,nv,valid,terminal,zeros,gamma=.95,lam=1.)
print('终止：advantage=',a['advantage'].tolist(),'return=',a['return_target'].tolist())
external=gae_with_boundaries(torch.zeros_like(r),v,torch.tensor([[.3,.8]],dtype=torch.float64),
                             valid,zeros,terminal,gamma=.95,lam=1.)
print('外部截断、可靠final_value：return=',external['return_target'].tolist())

终止：advantage= [[0.7499999999999999, 0.7]] return= [[0.95, 1.0]]
外部截断、可靠final_value：return= [[0.722, 0.76]]


## 10. 一张 RTX 3060 前的显存账本：先算下界

模型权重、参考模型、KV cache、激活、优化器、梯度、kernel工作空间、CUDA运行时各是不同项目。仅看某份模型的量化权重大小，不能直接断言GPU一定够用。

理想、未共享、无分配器额外开销的 KV cache：

\[
M_{KV}=2\times L\times B\times T\times H_{kv}\times D_h\times\mathrm{bytesPerElement}
\]

系数2来自K和V两份缓存。KV heads 与 Query heads 在GQA模型里不一定相同。生成过程的KV cache可复用前缀，节省重复计算，但需要显存。[Transformers KV cache](https://huggingface.co/docs/transformers/kv_cache)

仅作为示意：32层、8条并发、2048 token、8个KV头、head_dim128、每元素2字节，理想KV已达**2 GiB**。假设7B参数，仅fp16权重也约13.04 GiB；“4 bit乘0.5字节”只是理想化权重下界，还没算scale、参考、KV与激活。**没有实测3060，也没有承诺4bit后就能装下。**

本课小模型为了清楚直接重算前缀，实际上没有启用KV cache；下方计算器不是本课GPU profiler。

### 支持实现：KV缓存字节公式与明确遗漏项的容量下界

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [20]:
def kv_cache_bytes(layers:int,batch:int,sequence:int,kv_heads:int,head_dim:int,
                   bytes_per_element:int) -> int:
    for name,val in (("layers",layers),("batch",batch),("sequence",sequence),
                     ("kv_heads",kv_heads),("head_dim",head_dim),("bytes",bytes_per_element)):
        require(type(val) is int and 0 < val <= 10000000,name+"需要有界正整数")
    return 2*layers*batch*sequence*kv_heads*head_dim*bytes_per_element


def memory_lower_bounds(cfg:Config, model:TinyLM|None=None) -> dict:
    """理想KV的字节数与模型参数下界，绝非完整显存峰值。"""
    mib=1024**2
    scenario={"layers":32,"batch":8,"sequence":2048,"kv_heads":8,
              "head_dim":128,"bytes_per_element":2}
    kv=kv_cache_bytes(**scenario)
    def param_case(n,bytes_each):
        return n*bytes_each/mib
    results={"illustrative_7B":{
        "assumptions":{"parameters":7_000_000_000,"fp16_bytes":2,"naive_4bit_bytes":0.5,
                       "kv":scenario},
        "fp16_weights_gib":7_000_000_000*2/1024**3,
        "naive_4bit_weights_gib_lower_bound":7_000_000_000/2/1024**3,
        "ideal_kv_gib":kv/1024**3,
        "not_included":["quantization_scales_and_metadata","reference_model_weights_or_offload",
                        "activation_storage_and_attention_workspace","kernel_workspace_and_fragmentation",
                        "optimizer_states_and_gradient_buffers","generation_and_training_coexisting_buffers",
                        "CUDA_runtime_and_framework_overhead"],
        "warning":"仅示意下界；4bit不是承诺模型可装入3060，也不是设备实测"}}
    if model is not None:
        parameters=sum(p.numel() for p in model.parameters())
        results["tiny_model"]={"parameters":parameters,"fp32_weight_mib_lower_bound":param_case(parameters,4),
                               "train_batch":cfg.prompts_per_update*cfg.group_size,
                               "completion_cap":cfg.max_completion,
                               "note":"教学模型逐步重算全前缀，没有实际使用KV cache；下界非实测峰值"}
    return results

In [21]:
memory=memory_lower_bounds(CFG,small_ref)
print(json.dumps(memory,ensure_ascii=False,indent=2))
print('KV并发翻倍后的GiB:',kv_cache_bytes(32,16,2048,8,128,2)/1024**3)
print('KV长度翻倍后的GiB:',kv_cache_bytes(32,8,4096,8,128,2)/1024**3)

{
  "illustrative_7B": {
    "assumptions": {
      "parameters": 7000000000,
      "fp16_bytes": 2,
      "naive_4bit_bytes": 0.5,
      "kv": {
        "layers": 32,
        "batch": 8,
        "sequence": 2048,
        "kv_heads": 8,
        "head_dim": 128,
        "bytes_per_element": 2
      }
    },
    "fp16_weights_gib": 13.0385160446167,
    "naive_4bit_weights_gib_lower_bound": 3.259629011154175,
    "ideal_kv_gib": 2.0,
    "not_included": [
      "quantization_scales_and_metadata",
      "reference_model_weights_or_offload",
      "activation_storage_and_attention_workspace",
      "kernel_workspace_and_fragmentation",
      "optimizer_states_and_gradient_buffers",
      "generation_and_training_coexisting_buffers",
      "CUDA_runtime_and_framework_overhead"
    ],
    "warning": "仅示意下界；4bit不是承诺模型可装入3060，也不是设备实测"
  },
  "tiny_model": {
    "parameters": 19200,
    "fp32_weight_mib_lower_bound": 0.0732421875,
    "train_batch": 24,
    "completion_cap": 3,
    "note": "教学模

## 11. 完整训练：四个可审计的受控条件

每个条件从同一份SFT基座和LoRA初始A/B出发。固定18批新提示，每批6题×4次采样，批内两次更新；每个条件应实际生成 **432 条训练回答**，并单独记录有效EOS比例与各组零方差数。

四个分支不会因为同一采样种子就永远生成相同回答——参数更新后行为分布会改变。对照固定调度与seed，是控制可见变量，不是独立统计显著性证明。

value head另在独立的训练/开发前缀上学习Monte Carlo目标，不给任何分支提供优势。所有分支最后一步先保存，然后才对测试48题评估；没有根据测试结果回选最好的轮次。四个分支如果出现接近结果，也不能因为差异小就声称“不同公式等价”。

### 支持实现：SFT＋四分支token级GRPO实验、价值头、最终留出评测与重载

本节完整 Python 实现已嵌入讲义。先运行、观察随后的小实验，再按需要展开源码；无需导入配套 `.py`。

In [22]:
@torch.no_grad()
def eval_policy(model:TinyLM,tasks:list[dict],codec:CharCodec,cfg:Config,
                sample_seed:int,group_size:int=3) -> dict:
    require(bool(tasks),"评测任务不能为空")
    device=next(model.parameters()).device
    gold=[codec.encode(row["expected"])+[EOS] for row in tasks]
    batch=make_batch(tasks,gold,codec,cfg,device)
    lp=token_values(model(batch["ids"],batch["attention"]),batch,codec)
    exact=lp["sequence_logps"].exp().detach()
    samples=rollout(model,tasks,codec,cfg,torch.Generator(device=device).manual_seed(sample_seed),group_size)
    greedy=rollout(model,tasks,codec,cfg,torch.Generator(device=device).manual_seed(sample_seed),1,greedy=True)
    return {"tasks":len(tasks),"expected_success_probability":float(exact.mean()),
            "sample_successes":int(sum(x["reward"] for x in samples["samples"])),
            "sample_total":len(samples["samples"]),
            "greedy_successes":int(sum(x["reward"] for x in greedy["samples"])),
            "eos_fraction":sum(x["stop_reason"]=="eos" for x in samples["samples"])/len(samples["samples"]),
            "gold_probabilities": [{"id":r["id"],"p":float(p)} for r,p in zip(tasks,exact)],
            "samples":samples["samples"],"greedy":greedy["samples"]}


def slim_eval(result:dict)->dict:
    return {k:v for k,v in result.items() if k not in {"gold_probabilities","samples","greedy"}}


def full_experiment(base=Path("lesson_66_outputs"),cfg:Config|None=None,device_name="cpu",
                    arm_names:list[str]|None=None) -> Path:
    cfg=cfg or Config();cfg.validate()
    device=resolve_device(device_name)
    arms=arm_names or ["grpo","dapo","dr_grpo","dr_noscale"]
    require(arms and len(set(arms))==len(arms) and set(arms)<={"grpo","dapo","dr_grpo","dr_noscale"},
            "未知或重复训练分支")
    run_dir=Path(base)/(datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S_%fZ"))
    run_dir.mkdir(parents=True,exist_ok=False)
    manifest={"version":VERSION,"python":platform.python_version(),"torch":torch.__version__,
              "device":str(device),"status":"running","cfg":asdict(cfg),"sources":SOURCES,
              "scope":["synthetic_copy_task","token_level_critic_free_group_clipped_objectives",
                       "factorial_isolation_grpo_dapo_dr_with_same_group_scale_and_dr_noscale",
                       "separate_frozen_actor_MC_value_head","visited_state_KL_regularizer",
                       "no_pretrained_LLM_or_3060_measurements","no_complete_TRL_GRPO_reproduction"]}
    save_json(run_dir/"manifest.json",manifest)
    try:
        torch.set_num_threads(1)
        seed_all(cfg.seed)
        codec=CharCodec();tasks=make_tasks(cfg.seed)
        reference=TinyLM(len(codec.symbols),cfg).to(device)
        warmup=warmup_sft(reference,tasks["train"],codec,cfg)
        reference.requires_grad_(False).eval()
        frozen_ref=tensor_hash(reference.state_dict())
        # 保存起点的state，随后三个分支从完全相同LoRA初值复制。
        seed_all(cfg.seed+3)
        prototype=with_lora(reference,cfg).to(device)
        original_adapter=adapter_state(prototype)
        # 训练任务顺序固定；不把前轮样本喂给下一轮，或根据test选择参数。
        schedule_rng=random.Random(cfg.seed+13)
        schedule=[schedule_rng.sample(tasks["train"],cfg.prompts_per_update) for _ in range(cfg.updates)]
        save_json(run_dir/"tasks.json",tasks)
        save_json(run_dir/"schedule.json",[[x["id"] for x in row] for row in schedule])
        save_json(run_dir/"codec.json",codec.contract())
        save_json(run_dir/"memory_estimates.json",memory_lower_bounds(cfg,reference))
        save_json(run_dir/"sft.json",warmup)
        critic_report,critic_weights=train_critic(reference,tasks,codec,cfg)
        save_json(run_dir/"critic.json",critic_report)
        torch.save({"version":VERSION,"cfg":asdict(cfg),"state":{n:t.cpu().clone() for n,t in reference.state_dict().items()},
                    "codec":codec.contract(),"reference_hash":frozen_ref,"critic":critic_weights},
                   run_dir/"reference.pt")
        result={"cfg":asdict(cfg),"model_parameters":sum(p.numel() for p in reference.parameters()),
                "adapter_parameters":sum(p.numel() for p in prototype.parameters() if p.requires_grad),
                "initial_adapter_hash":tensor_hash(original_adapter),
                "reference_hash":frozen_ref,"arms":{},"critic":critic_report,
                "no_test_driven_selection":True}
        for name in arms:
            policy=copy.deepcopy(prototype).to(device)
            opt=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=cfg.rl_lr)
            rng=torch.Generator(device=device).manual_seed(cfg.seed+105)
            history=[];actual_updates=0;all_rollouts=[]
            scaling="none" if name=="dr_noscale" else "group"
            normalization="dr_grpo" if name=="dr_noscale" else name
            print("训练token分支:",name,flush=True)
            for i,rows in enumerate(schedule):
                roll=rollout(policy,rows,codec,cfg,rng)
                fixed=align_rollout(roll,policy,reference,codec,cfg,scaling)
                losses=[]
                for _ in range(cfg.inner_epochs):
                    info=grpo_update(policy,reference,fixed,codec,cfg,opt,normalization)
                    losses.append(info)
                    actual_updates+=int(info["updated"])
                history.append({"batch":i+1,"reward":float(fixed["rewards"].mean()),
                                "zero_variance_groups":sum(len(set(roll["samples"][j*cfg.group_size+k]["reward"]
                                       for k in range(cfg.group_size)))==1 for j in range(cfg.prompts_per_update)),
                                "eos_fraction":sum(s["stop_reason"]=="eos" for s in roll["samples"])/len(roll["samples"]),
                                "steps":losses})
                all_rollouts.append({"batch":i+1,"policy_hash":roll["policy_hash"],
                                     "samples":roll["samples"],"advantages":fixed["advantages"].tolist()})
            require(tensor_hash(reference.state_dict())==frozen_ref,"参考模型意外改变")
            require(base_hash(policy)==frozen_ref,"LoRA基座意外改变")
            require(all(p.grad is None for p in reference.parameters()),"参考收到梯度")
            torch.save({"version":VERSION,"arm":name,"rank":cfg.lora_rank,"alpha":cfg.lora_alpha,
                        "reference_hash":frozen_ref,"codec":codec.contract(),
                        "adapter":adapter_state(policy)},run_dir/(name+".pt"))
            save_jsonl(run_dir/(name+"_rollouts.jsonl"),all_rollouts)
            result["arms"][name]={"loss_type":normalization,"reward_scaling":scaling,
                                  "updates":actual_updates,"rollouts":len(history),
                                  "completed_sequences":sum(len(row["samples"]) for row in all_rollouts),
                                  "generated_answer_tokens":sum(len(s["token_ids"]) for row in all_rollouts for s in row["samples"]),
                                  "history":history,"reference_frozen":True,"base_frozen":True}
            # 模型留在CPU直到test评估，测试仅在所有分支训练和保存后执行。
            if name == arms[0]:
                all_policy={}
            all_policy[name]=policy.cpu()
        evaluation={"reference":eval_policy(reference,tasks["test"],codec,cfg,cfg.seed+607)}
        for name,model in all_policy.items():
            evaluation[name]=eval_policy(model.to(device),tasks["test"],codec,cfg,cfg.seed+607)
            result["arms"][name]["test"]=slim_eval(evaluation[name])
        result["reference_test"]=slim_eval(evaluation["reference"])
        save_json(run_dir/"evaluation.json",evaluation)
        save_json(run_dir/"training.json",result)
        reload_result=reload_experiment(run_dir,device_name)
        save_json(run_dir/"reload.json",reload_result)
        manifest["status"]="completed";save_json(run_dir/"manifest.json",manifest)
        print("结果目录:",run_dir.resolve(),flush=True)
        return run_dir
    except Exception as exc:
        manifest.update(status="failed",error=type(exc).__name__+": "+str(exc))
        save_json(run_dir/"manifest.json",manifest)
        raise


def reload_experiment(run_dir:Path,device_name="cpu") -> dict:
    run_dir=Path(run_dir)
    device=resolve_device(device_name)
    base=torch.load(run_dir/"reference.pt",map_location="cpu",weights_only=True)
    require(base["version"]==VERSION,"版本不匹配")
    cfg=Config(**base["cfg"]);cfg.validate()
    codec=CharCodec();require(base["codec"]==codec.contract(),"tokenizer/动作集合不匹配")
    model=TinyLM(len(codec.symbols),cfg).to(device)
    model.load_state_dict(base["state"],strict=True);model.requires_grad_(False).eval()
    require(tensor_hash(model.state_dict())==base["reference_hash"],"参考权重与摘要不符")
    tasks=json.loads((run_dir/"tasks.json").read_text(encoding="utf-8"))
    validate_tasks(tasks)
    report=json.loads((run_dir/"evaluation.json").read_text(encoding="utf-8"))
    out={}
    for name in sorted(set(report)-{"reference"}):
        blob=torch.load(run_dir/(name+".pt"),map_location="cpu",weights_only=True)
        require(blob["version"]==VERSION and blob["reference_hash"]==base["reference_hash"],
                "adapter绑定错误")
        require(blob["codec"]==codec.contract(),"adapter词表不匹配")
        p=with_lora(model,cfg)
        load_adapter(p,blob["adapter"]);p.requires_grad_(False).eval()
        result=eval_policy(p,tasks["test"],codec,cfg,cfg.seed+607)
        original=report[name]
        max_error=max(abs(x["p"]-y["p"]) for x,y in zip(result["gold_probabilities"],original["gold_probabilities"]))
        ids_ok=[r["token_ids"] for r in result["greedy"]]==[r["token_ids"] for r in original["greedy"]]
        require(max_error<3e-5 and ids_ok,"重载评估不一致；检查后端与版本")
        out[name]={"max_probability_error":max_error,"greedy_ids_equal":ids_ok,
                   "trainable_parameters":sum(x.numel() for x in p.parameters() if x.requires_grad)}
    return {"arms":out,"mode":"inference_reload_only; not exact training resume"}

In [23]:
RUN_DIRECTORY = None
if RUN_FULL:
    RUN_DIRECTORY = full_experiment(OUTPUT_ROOT,CFG,DEVICE)
    print('本次完整实验目录：',RUN_DIRECTORY)
else:
    print('本次明确跳过完整训练；后面不会填造训练数字。')

训练token分支: grpo


训练token分支: dapo


训练token分支: dr_grpo


训练token分支: dr_noscale


结果目录: /mnt/data/_lesson66_build/notebook_run/lesson_66_outputs/run_20261103_123457_459667Z


本次完整实验目录： lesson_66_outputs/run_20261103_123457_459667Z


## 12. 分开看固定完整轨迹概率、实际采样与贪心

本课每题只有一条合格轨迹：两位正确数字＋EOS，因此可以直接计算**这条轨迹概率**，作为受限三步模型的精确任务成功概率。复杂自然语言题有多种正确表达，不能用一个参考序列的概率代替所有正确答案概率。

实际有限采样还受到随机波动影响；贪心选择是另一种生成决策。不管哪项提高，都不能包装成“基座LLM通用推理能力提高”。

In [24]:
if RUN_DIRECTORY is not None:
    TRAINING=json.loads((RUN_DIRECTORY/'training.json').read_text(encoding='utf-8'))
    EVALUATION=json.loads((RUN_DIRECTORY/'evaluation.json').read_text(encoding='utf-8'))
    print('方法           固定轨迹成功概率   实际成功/采样数   贪心正确/48')
    for name,r in EVALUATION.items():
        print(f"{name:14} {r['expected_success_probability']:.6f}           "
              f"{r['sample_successes']}/{r['sample_total']}            {r['greedy_successes']}/48")
    print('各分支的归约分母与奖励缩放：')
    for name,r in TRAINING['arms'].items():
        print(name,r['loss_type'],r['reward_scaling'],'更新次数=',r['updates'])
else:
    print('未运行本次训练，不显示预填成绩。')

方法           固定轨迹成功概率   实际成功/采样数   贪心正确/48
dapo           0.192912           32/144            31/48
dr_grpo        0.193001           32/144            31/48
dr_noscale     0.192464           32/144            31/48
grpo           0.188739           32/144            29/48
reference      0.153170           26/144            22/48
各分支的归约分母与奖励缩放：
dapo dapo group 更新次数= 36
dr_grpo dr_grpo group 更新次数= 36
dr_noscale dr_grpo none 更新次数= 36
grpo grpo group 更新次数= 36


### 一次真实 CPU 实验的结果该怎样阅读？

制作环境里的默认参数得到：SFT参考的轨迹概率约0.153，GRPO型约0.189，DAPO型与两种Dr. GRPO型约0.193。贪心题数由22/48分别变为29/48和31/48。价值头的训练与开发MSE也下降。

**这些数字来自该份随机小字符模型**，没有预训练权重、没有多seed泛化保证、没有实际测量GPU吞吐或峰值显存。训练目标包含人工规则奖励；单次实验不能据此宣布哪种GRPO分母普遍更优。你本机重新训练的文件以实际输出为准。

更重要的是：`grpo`、`dapo`、`dr_grpo`同样使用组内std，可观察归约分母的差异；`dr_grpo`与`dr_noscale`同一分母，可以观察奖励缩放差异。这是变量隔离，不是把四条训练报告排名当成因果定论。

In [25]:
if RUN_DIRECTORY is not None:
    critic_report=TRAINING['critic']
    print('独立ValueHead：训练MSE',critic_report['before']['train_mse'],'→',critic_report['after']['train_mse'])
    print('独立ValueHead：开发MSE',critic_report['before']['dev_mse'],'→',critic_report['after']['dev_mse'])
    print('真实训练样本统计：')
    for name,arm in TRAINING['arms'].items():
        print(name,{'完成序列':arm['completed_sequences'], '输出token':arm['generated_answer_tokens'],
                    '实际LoRA更新':arm['updates'],
                    '全同分组总数':sum(row['zero_variance_groups'] for row in arm['history'])})

独立ValueHead：训练MSE 0.32522353529930115 → 0.1626264750957489
独立ValueHead：开发MSE 0.2738865613937378 → 0.11662016063928604
真实训练样本统计：
dapo {'完成序列': 432, '输出token': 1278, '实际LoRA更新': 36, '全同分组总数': 45}
dr_grpo {'完成序列': 432, '输出token': 1278, '实际LoRA更新': 36, '全同分组总数': 45}
dr_noscale {'完成序列': 432, '输出token': 1278, '实际LoRA更新': 36, '全同分组总数': 45}
grpo {'完成序列': 432, '输出token': 1282, '实际LoRA更新': 36, '全同分组总数': 46}


## 13. 保存、缓存与重新加载的边界

完整运行的主要文件：

| 文件 | 内容 |
|---|---|
| `reference.pt` | SFT参考与独立ValueHead的真实张量、结构与身份 |
| `grpo.pt`、`dapo.pt`、`dr_grpo.pt`、`dr_noscale.pt` | 每个分支的A/B、rank、alpha及固定参考身份 |
| `*_rollouts.jsonl` | 实际token、停止状态、奖励、优势、来源与批次 |
| `training.json`、`evaluation.json` | 可复核的训练与最终测试指标 |
| `critic.json` | 独立价值头的训练/开发损失 |
| `memory_estimates.json` | 具有明确假设和遗漏项的显存下界 |
| `reload.json` | 重新加载后的概率与贪心token一致性 |

新进程重载使用 `weights_only=True`，核对base与adapter身份，并在相同数据/seed上重新生成。不从不可信文件加载权重；摘要仅能发现意外错配，不是来源签名。没有保存优化器、完整随机状态与进度游标，所以**这不是精确断点续训**。

In [26]:
if RUN_DIRECTORY is not None:
    print(json.dumps(reload_experiment(RUN_DIRECTORY,DEVICE),ensure_ascii=False,indent=2))

{
  "arms": {
    "dapo": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "dr_grpo": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "dr_noscale": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    },
    "grpo": {
      "max_probability_error": 0.0,
      "greedy_ids_equal": true,
      "trainable_parameters": 0
    }
  },
  "mode": "inference_reload_only; not exact training resume"
}


## 14. 完整自动检查：先判断机制，再谈效果

本课固定检查会核对：EOS/格式/奖励，prompt和padding mask，采样概率对齐，四种归约的梯度，组内全零优势，参考KL，GAE真实终止与外部截断，KV字节公式，训练参数冻结、价值头训练和独立短训练重载。

通过仅代表本课**有界合成任务、CPU微型模型、明确公式和输入契约**。它不能代替完整框架互操作、GPU性能、安全审查或真实LLM效果评测。

In [27]:
class Checks:
    def __init__(self): self.names=[]
    def ok(self,name,condition):
        if not bool(condition): raise AssertionError(name)
        self.names.append(name)
    def close(self,name,actual,expected,atol=1e-7,rtol=1e-7):
        a=torch.as_tensor(actual,dtype=torch.float64)
        b=torch.as_tensor(expected,dtype=torch.float64)
        self.ok(name,a.shape==b.shape and torch.allclose(a,b,atol=atol,rtol=rtol))
    def rejects(self,name,fn):
        try:fn()
        except (LessonError,ValueError,TypeError):self.names.append(name);return
        raise AssertionError(name+" 没有拒绝")


def check_exercise_a(fn):
    cfg=Config(); codec=CharCodec();tasks=make_tasks()["train"][:2]
    ans=[codec.encode(tasks[0]["expected"])+[EOS],[EOS]]
    batch=make_batch(tasks,ans,codec,cfg)
    logits=torch.randn(2,batch["ids"].shape[1],len(codec.symbols),
                       dtype=torch.float64,requires_grad=True)
    data=token_values(logits,batch,codec)
    expected=extract_completion_positions(data,cfg)
    got=fn(data,cfg)
    for key in ("token_logps","mask","counts","token_distribution"):
        if key=="mask":
            require(torch.equal(got[key],expected[key]),"A: EOS/pad mask错误")
        else:
            require(torch.allclose(got[key],expected[key]),"A: token收集/归一化错误")
    g1=torch.autograd.grad(got["token_logps"].sum(),logits,retain_graph=True)[0]
    g2=torch.autograd.grad(expected["token_logps"].sum(),logits)[0]
    require(torch.allclose(g1,g2),"A: 梯度位置错误")
    return {"exercise":"A","passed":True}


def check_exercise_b(fn):
    rewards=torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]],dtype=torch.float64,requires_grad=True)
    for mode in ("group","none"):
        got=fn(rewards,mode)
        ref=group_advantages(rewards,mode)
        require(torch.allclose(got,ref,atol=1e-12),"B: 组优势错误")
        require(not got.requires_grad,"B: 组奖励不应反传")
        require(torch.equal(rewards.detach(),torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]],dtype=torch.float64)),
                "B: 不应改输入")
    return {"exercise":"B","passed":True}


def check_exercise_c(fn):
    old=torch.zeros(2,3,dtype=torch.float64,requires_grad=True)
    cur=torch.tensor([[0.,0.,0.],[0.,0.,0.]],dtype=torch.float64,requires_grad=True)
    mask=torch.tensor([[1,0,0],[1,1,1]],dtype=torch.bool)
    a=torch.tensor([1.,-1.],dtype=torch.float64,requires_grad=True)
    for mode in ("grpo","dapo","dr_grpo"):
        expected,_=token_clip_loss(cur,old,mask,a,.2,mode,3)
        got=fn(cur,old,mask,a,.2,mode,3)
        require(torch.allclose(got,expected,atol=1e-12),"C: 三种归约有误")
        gn,go,ga=torch.autograd.grad(got,(cur,old,a),retain_graph=True,allow_unused=True)
        true_grad=torch.autograd.grad(expected,cur,retain_graph=True)[0]
        require(torch.allclose(gn,true_grad,atol=1e-12),"C: 当前logp梯度错误")
        require(go is None and ga is None,"C: 旧策略/优势不应有梯度")
    # 溢出区间才可检出 min 被换成 max 的故障。
    extreme=torch.tensor([[math.log(1.6),0.,0.],[math.log(.5),0.,0.]],
                         dtype=torch.float64,requires_grad=True)
    emask=torch.tensor([[True,False,False],[True,False,False]])
    for mode in ("grpo","dapo","dr_grpo"):
        target,_=token_clip_loss(extreme,old,emask,a,.2,mode,3)
        result=fn(extreme,old,emask,a,.2,mode,3)
        require(torch.allclose(result,target,atol=1e-12),"C: 裁剪边界与min方向错误")
        g1=torch.autograd.grad(result,extreme,retain_graph=True)[0]
        g2=torch.autograd.grad(target,extreme,retain_graph=True)[0]
        require(torch.allclose(g1,g2,atol=1e-12),"C: 裁剪平坦区梯度错误")
    return {"exercise":"C","passed":True}

## 15. 独立练习 A：只收集回答token

实现 `pack_ex(data,cfg)`：根据有效token mask，将每条样本的token log概率收集成 `[N,Lmax]`。不修改提示，EOS保留、padding补零；完整动作log分布的padding填入归一化均匀分布；输出包含 `mask` 与 `counts`。

检查器会比较每个位置及**完整 logits 梯度**，防止用 `.tolist()` 或 `.detach()` 偷偷切断计算图。

In [28]:
RUN_EXERCISE_A = False

def pack_ex(data,cfg):
    raise NotImplementedError('练习A：实现有效回答位置的token收集')

if RUN_EXERCISE_A:
    print(check_exercise_a(pack_ex))

## 16. 独立练习 B：组相对优势

实现 `group_adv_ex(rewards,mode)`，输入 `[B,G]`。`mode='group'`按总体标准差缩放，`mode='none'`只中心化。不能原地修改奖励；要停止梯度；全同分组返回零，不要除零。

检查器使用一组 `[1,0,0,0]` 与一组全1结果，并检查输入未被修改。

In [29]:
RUN_EXERCISE_B = False

def group_adv_ex(rewards,mode):
    raise NotImplementedError('练习B：实现同题组内优势')

if RUN_EXERCISE_B:
    print(check_exercise_b(group_adv_ex))

## 17. 独立练习 C：三种不同长度归约

实现 `clip_loss_ex(new,old,mask,a,eps,variant,max_len)`。逐token ratio先计算，正负优势采用正确的 `min`；无效位置不计入分子/分母。支持 `grpo`、`dapo` 与 `dr_grpo` 三种归约，并且只向当前 logp 返回梯度。

测试故意构造两条长度1与3的回答，且测试越界比例的平坦梯度；**只在ratio=1时测试，将检测不出 `min` 误写成 `max`**。

In [30]:
RUN_EXERCISE_C = False

def clip_loss_ex(new,old,mask,a,eps,variant,max_len):
    raise NotImplementedError('练习C：实现clip与三种归约')

if RUN_EXERCISE_C:
    print(check_exercise_c(clip_loss_ex))

## 18. 参考实现与故障注入

先独立完成练习，再展开参考答案。函数独立命名，不覆盖你的实现。

接着故意尝试：PAD进入监督、使用样本std代替总体std、`min`换成`max`、合并不同归约、cap误判为EOS、外部截断丢bootstrap、只计算K不计算V。测试应真正失败，才算检出；并非“穷尽漏洞”的证明。

In [31]:
def reference_a(data,cfg):
    valid=data["mask"]
    lp=data["token_logps"]
    dist=data["token_distribution"]
    rows_lp=[];rows_dist=[];rows_mask=[]
    for row in range(lp.shape[0]):
        k=int(valid[row].sum())
        rows_lp.append(F.pad(lp[row][valid[row]],(0,cfg.max_completion-k)))
        rows_dist.append(F.pad(dist[row][valid[row]],(0,0,0,cfg.max_completion-k),
                               value=-math.log(dist.shape[-1])))
        rows_mask.append(torch.arange(cfg.max_completion,device=lp.device)<k)
    return {"token_logps":torch.stack(rows_lp),"token_distribution":torch.stack(rows_dist),
            "mask":torch.stack(rows_mask),"counts":valid.sum(1)}


def reference_b(rewards,mode):
    frozen=rewards.detach()
    result=frozen-frozen.mean(dim=1,keepdim=True)
    if mode=="group":
        result=result/(frozen.std(dim=1,keepdim=True,unbiased=False)+1e-6)
    return result


def reference_c(new,old,mask,advantages,epsilon,variant,max_completion):
    r=torch.exp((new-old.detach()).masked_fill(~mask,0))
    a=advantages.detach().unsqueeze(1)
    values=torch.minimum(r*a,r.clamp(1-epsilon,1+epsilon)*a).masked_fill(~mask,0)
    if variant=="grpo":return -(values.sum(1)/mask.sum(1).clamp_min(1)).mean()
    if variant=="dapo":return -values.sum()/mask.sum()
    if variant=="dr_grpo":return -values.sum()/(new.shape[0]*max_completion)
    raise LessonError("无效归约")

In [32]:
print('A:',check_exercise_a(reference_a))
print('B:',check_exercise_b(reference_b))
print('C:',check_exercise_c(reference_c))

A: {'exercise': 'A', 'passed': True}
B: {'exercise': 'B', 'passed': True}
C: {'exercise': 'C', 'passed': True}


In [33]:
def run_checks() -> dict:
    torch.set_num_threads(1)
    seed_all(6610)
    c=Checks();cfg=Config(sft_steps=3,updates=2,critic_steps=10)
    codec=CharCodec();splits=make_tasks();validate_tasks(splits)
    c.ok("固定160/48/48分组",[len(splits[k]) for k in ("train","dev","test")]==[160,48,48])
    c.ok("词表20符号、11回答动作",len(codec.symbols)==20 and len(codec.answer_ids)==11)
    c.ok("任务标签不进入prompt字段",all("expected" not in r["prompt"] for r in splits["train"]))
    c.ok("训练测试提示完全不交叉",not {r["prompt"] for r in splits["train"]}&{r["prompt"] for r in splits["test"]})
    c.ok("EOS正确结果",score_completion("02","eos","02")["reward"]==1)
    c.ok("任务cap没有EOS不是完整答案",score_completion("02","horizon","02")["reward"]==0)
    c.ok("前导零必须保留",score_completion("2","eos","02")["reward"]==0)
    c.ok("非法格式奖励0",score_completion("0a","eos","02")["reason"]=="invalid_format")
    c.rejects("外部运行故障不能伪造零奖励",lambda:score_completion("02","interrupted","02"))
    c.rejects("未定义答案词表拒绝",lambda:codec.encode("中文"))
    c.rejects("batch<2的组奖励拒绝",lambda:group_advantages(torch.ones(2,1)))
    c.rejects("非法group scaling拒绝",lambda:group_advantages(torch.ones(1,2),"bad"))
    c.rejects("非法Config group拒绝",lambda:Config(group_size=1).validate())
    c.rejects("非法Config权重拒绝",lambda:Config(ref_kl_coef=-1).validate())
    c.rejects("非法Config字符预算拒绝",lambda:Config(max_completion=9).validate())
    data=torch.tensor([[1.,0.,0.,0.],[1.,1.,1.,1.]],dtype=torch.float64,requires_grad=True)
    std_adv=group_advantages(data)
    c.close("组内单位方差总体标准差",std_adv[0],[math.sqrt(3),-1/math.sqrt(3),-1/math.sqrt(3),-1/math.sqrt(3)],atol=1e-5)
    c.close("同分组无信号",std_adv[1],[0,0,0,0])
    c.ok("组优势不反向传播",not std_adv.requires_grad)
    c.close("不缩放只中心化",group_advantages(data,"none")[0],[.75,-.25,-.25,-.25])
    c.close("组内求和约零",std_adv.sum(1),[0,0],atol=1e-6)
    sample=splits["train"][:2]
    ans=[codec.encode(sample[0]["expected"])+[EOS],[EOS]]
    batch=make_batch(sample,ans,codec,cfg)
    pfx=len(sample[0]["prompt"])+1
    c.ok("prompt监督标签全忽略",bool((batch["labels"][:,:pfx]==IGNORE).all()))
    c.ok("两个EOS均计入监督",int((batch["labels"]==EOS).sum())==2)
    c.ok("PAD标签忽略",bool((batch["labels"][1,-2:]==IGNORE).all()))
    c.rejects("EOS后多余token拒绝",lambda:make_batch(sample[:1],[[EOS,EOS]],codec,cfg))
    c.rejects("静默超长拒绝",lambda:make_batch(sample,ans,codec,Config(max_context=12)))
    logits=torch.zeros(2,batch["ids"].shape[1],len(codec.symbols),dtype=torch.float64,requires_grad=True)
    full=token_values(logits,batch,codec)
    packed=extract_completion_positions(full,cfg)
    c.close("uniform 11动作首行3个token",packed["token_logps"][0],[-math.log(11)]*3)
    c.close("uniform第二行EOS后补零",packed["token_logps"][1],[-math.log(11),0,0])
    c.close("count目标3和1",packed["counts"],[3,1])
    c.ok("mask包含EOS但不含其后补齐",packed["mask"].tolist()==[[True,True,True],[True,False,False]])
    grad=torch.autograd.grad(packed["token_logps"].sum(),logits)[0]
    c.ok("提示logits不直接被监督",bool((grad[:,:pfx-1]==0).all()))
    c.ok("首回答前一位置有梯度",float(grad[:,pfx-1].abs().sum())>0)
    c.ok("填充位置梯度为零",float(grad[1,-2:].abs().sum())==0)
    c.close("padding分布仍正规化",packed["token_distribution"].exp().sum(-1),torch.ones(2,3,dtype=torch.float64))
    c.ok("BOS是合法attention但不是loss",bool((batch["attention"][:,0]).all()) and bool((batch["labels"][:,0]==IGNORE).all()))
    c.rejects("非法输出词表尺寸拒绝",lambda:action_distribution(torch.zeros(1,10),codec))
    c.rejects("非有限logits拒绝",lambda:action_distribution(torch.full((1,20),float('nan')),codec))
    c.ok("当前与参考分布的KL零",float(visited_categorical_kl(packed["token_distribution"],
             packed["token_distribution"].detach(),packed["mask"]).detach())<1e-7)
    shifted=packed["token_distribution"].detach().clone()
    shifted[0,0,0] += 1
    shifted[0,0] = F.log_softmax(shifted[0,0],-1)
    c.ok("变化的参考分布KL严格正",float(visited_categorical_kl(packed["token_distribution"],shifted,packed["mask"]).detach())>0)
    c.rejects("未归一化分布不可以算KL",lambda:visited_categorical_kl(packed["token_distribution"],shifted+2,packed["mask"]))
    n=torch.zeros(2,3,dtype=torch.float64,requires_grad=True)
    old=torch.zeros_like(n,requires_grad=True)
    valid=torch.tensor([[True,False,False],[True,True,True]])
    a=torch.tensor([1.,-1.],dtype=torch.float64,requires_grad=True)
    expectations={"grpo":0.,"dapo":.5,"dr_grpo":1/3}
    for method,expected in expectations.items():
        loss,info=token_clip_loss(n,old,valid,a,.2,method,3)
        c.close("归约分母不同_"+method,loss,expected)
        g,go,ga=torch.autograd.grad(loss,(n,old,a),retain_graph=True,allow_unused=True)
        c.ok("仅当前策略得到梯度_"+method,go is None and ga is None and bool(torch.isfinite(g).all()))
        c.ok("有效token与completion正确_"+method,info["active_tokens"]==4 and info["eligible_completions"]==2)
    outside=torch.tensor([[math.log(1.6),0.,0.],[math.log(.5),0.,0.]],dtype=torch.float64,requires_grad=True)
    tmask=torch.tensor([[True,False,False],[True,False,False]])
    clipped,notes=token_clip_loss(outside,torch.zeros_like(outside),tmask,a,.2,"grpo",3)
    clipped_g=torch.autograd.grad(clipped,outside)[0]
    c.close("正优势超上限梯度为零",clipped_g[0,0],0.)
    c.close("负优势低于下限梯度为零",clipped_g[1,0],0.)
    c.close("裁剪方向实现",notes["clip_fraction"],1.)
    filtered,meta=token_clip_loss(n,old,valid,a,.2,"grpo",3,
                                  torch.tensor([True,True]))
    c.ok("全截断过滤返回skipped",meta["skipped"] and meta["active_tokens"]==0)
    c.close("全截断过滤无假梯度",torch.autograd.grad(filtered,n)[0],torch.zeros_like(n))
    c.rejects("无效变体拒绝",lambda:token_clip_loss(n,old,valid,a,.2,"wrong",3))
    c.rejects("非法epsilon拒绝",lambda:token_clip_loss(n,old,valid,a,1.,"grpo",3))
    c.rejects("非有限优势拒绝",lambda:token_clip_loss(n,old,valid,torch.tensor([float('nan'),0.]),.2,"grpo",3))

    r=torch.tensor([[0.,1.]],dtype=torch.float64)
    v=torch.tensor([[.2,.3]],dtype=torch.float64)
    nv=torch.tensor([[.3,0.]],dtype=torch.float64)
    mask=torch.tensor([[True,True]]);term=torch.tensor([[False,True]]);trunc=torch.zeros_like(term)
    ga=gae_with_boundaries(r,v,nv,mask,term,trunc,gamma=.95,lam=1.)
    c.close("真实终局return",ga["return_target"],[[.95,1.]])
    c.close("真实终局GAE",ga["advantage"],[[.75,.7]])
    v_ext=torch.tensor([[.3,.8]],dtype=torch.float64)
    term_ext=torch.tensor([[False,False]]);trunc_ext=torch.tensor([[False,True]])
    rg=torch.tensor([[0.,0.]],dtype=torch.float64)
    external=gae_with_boundaries(rg,v,v_ext,mask,term_ext,trunc_ext,gamma=.95,lam=1.)
    c.close("外部截断应bootstrap到final_value",external["return_target"],[[.722,.76]])
    wrong=gae_with_boundaries(rg,v,v_ext,mask,term,trunc,gamma=.95,lam=1.)
    c.ok("外部截断当terminal会改变return",float((external["return_target"]-wrong["return_target"]).abs().max())>.5)
    c.rejects("同一位置不能双边界标记",lambda:gae_with_boundaries(r,v,nv,mask,term,term))
    c.rejects("末端必须有边界",lambda:gae_with_boundaries(r,v,nv,mask,torch.zeros_like(term),torch.zeros_like(term)))
    c.rejects("非法gamma拒绝",lambda:gae_with_boundaries(r,v,nv,mask,term,trunc,gamma=1.2))
    c.ok("GAE不反向到值预测",not ga["advantage"].requires_grad)

    scenario=memory_lower_bounds(cfg)["illustrative_7B"]
    c.close("理想KV样例2GiB",scenario["ideal_kv_gib"],2.)
    kv=kv_cache_bytes(32,8,2048,8,128,2)
    c.ok("KV随batch线性增加",kv_cache_bytes(32,16,2048,8,128,2)==2*kv)
    c.ok("KV随seq线性增加",kv_cache_bytes(32,8,4096,8,128,2)==2*kv)
    c.rejects("KV负batch拒绝",lambda:kv_cache_bytes(32,-1,2048,8,128,2))
    c.rejects("KV布尔batch拒绝",lambda:kv_cache_bytes(32,True,2048,8,128,2))
    c.ok("4bit只是权重下界不是适配结论",scenario["naive_4bit_weights_gib_lower_bound"]<scenario["fp16_weights_gib"]
         and len(scenario["not_included"])>=5)

    model=TinyLM(len(codec.symbols),cfg).eval()
    ref=copy.deepcopy(model).requires_grad_(False).eval()
    policy=with_lora(ref,cfg).eval()
    c.ok("LoRA与参考起点同输出",torch.equal(policy(batch["ids"],batch["attention"]),
                                              ref(batch["ids"],batch["attention"])))
    c.ok("参考独立存储",model.tokens.weight.data_ptr()!=ref.tokens.weight.data_ptr())
    c.ok("只A/B训练",all(p.requires_grad==n.endswith((".A",".B")) for n,p in policy.named_parameters()))
    c.ok("基座与参考哈希可比",base_hash(policy)==tensor_hash(ref.state_dict()))
    # 固定输出模型验证两个终止状态；不要随机依赖样本刚好抽到EOS。
    class Fixed(nn.Module):
        def __init__(self,token_idx):
            super().__init__();self.dummy=nn.Parameter(torch.zeros(1),requires_grad=False)
            self.token_idx=token_idx
        def forward(self,ids,valid=None):
            out=torch.zeros(ids.shape[0],ids.shape[1],len(codec.symbols),device=ids.device)
            out[:,:,self.token_idx]=30
            return out
    eos_roll=rollout(Fixed(EOS),sample,codec,cfg,torch.Generator().manual_seed(1),greedy=True)
    c.ok("立即EOS算终止并得零奖励",all(s["stop_reason"]=="eos" and s["token_ids"]==[EOS] and s["reward"]==0
                                             for s in eos_roll["samples"]))
    digit_roll=rollout(Fixed(codec.ids["1"]),sample,codec,cfg,torch.Generator().manual_seed(1),greedy=True)
    c.ok("未EOS到cap算任务结束",all(s["stop_reason"]=="horizon" and len(s["token_ids"])==3 for s in digit_roll["samples"]))
    rng=torch.Generator().manual_seed(66)
    sampled=rollout(policy,sample,codec,cfg,rng,group_size=cfg.group_size)
    c.ok("真实采样保留全部组槽",len(sampled["samples"])==2*cfg.group_size)
    c.ok("每条回答不包含EOS后token",all(EOS not in x[:-1] for x in sampled["answers"]))
    c.ok("逐步采样logp数量等于token数",all(len(x["sampled_logps"])==len(x["token_ids"]) for x in sampled["samples"]))
    c.ok("改变expected不影响实际采样",sampled["answers"]==rollout(policy,
        [dict(item,expected="99") for item in sample],codec,cfg,
        torch.Generator().manual_seed(66),group_size=cfg.group_size)["answers"])
    fixed=align_rollout(sampled,policy,ref,codec,cfg,"group")
    c.ok("旧概率不带梯度",not fixed["old_logps"].requires_grad)
    c.ok("组优势为同题分组",len(fixed["advantages"])==cfg.group_size*2)
    c.ok("旧概率PAD部分为零",bool((fixed["old_logps"][~fixed["mask"]]==0).all()))
    with torch.no_grad():
        out=extract_completion_positions(token_values(policy(fixed["batch"]["ids"],fixed["batch"]["attention"]),
                       fixed["batch"],codec),cfg)
    c.close("新采样起点ratio为1",out["token_logps"][fixed["mask"]],fixed["old_logps"][fixed["mask"]],atol=3e-5)
    ref_before=tensor_hash(ref.state_dict());base_before=base_hash(policy)
    optim=torch.optim.Adam([p for p in policy.parameters() if p.requires_grad],lr=.002)
    data=grpo_update(policy,ref,fixed,codec,cfg,optim,"grpo")
    c.ok("LoRA真实更新一步",data["updated"] and fixed["updated"]==1)
    c.ok("冻结基座未变",base_hash(policy)==base_before)
    c.ok("冻结参考未变",tensor_hash(ref.state_dict())==ref_before)
    c.ok("参考无grad",all(p.grad is None for p in ref.parameters()))
    c.ok("loss与梯度有限",math.isfinite(data["total_loss"]) and math.isfinite(data["grad_norm_before_clip"]))
    c.ok("旧概率在更新后不变",not fixed["old_logps"].requires_grad)
    c.rejects("重复注入LoRA拒绝",lambda:with_lora(policy,cfg))
    c.rejects("测试样本不许进入训练",lambda:align_rollout(rollout(policy,splits["test"][:1],codec,cfg,
       torch.Generator().manual_seed(66),group_size=2),policy,ref,codec,cfg,"group"))
    critic_before=tensor_hash(ref.state_dict())
    val,val_state=train_critic(ref,splits,codec,Config(critic_steps=12,sft_steps=3))
    c.ok("价值头训练集误差下降",val["after"]["train_mse"]<val["before"]["train_mse"])
    c.ok("独立价值头不修改actor",tensor_hash(ref.state_dict())==critic_before)
    c.ok("价值头有独立参数",val["critic_parameters"]==cfg.dim+1 and set(val_state)=={"linear.weight","linear.bias"})
    c.ok("GAE与组优势不同实现入口",bool(gae_with_boundaries(r,v,nv,mask,term,trunc)))
    c.ok("练习A参考通过",check_exercise_a(reference_a)["passed"])
    c.ok("练习B参考通过",check_exercise_b(reference_b)["passed"])
    c.ok("练习C参考通过",check_exercise_c(reference_c)["passed"])
    with tempfile.TemporaryDirectory() as temp:
        short=Config(sft_steps=2,updates=2,critic_steps=4)
        path=full_experiment(Path(temp),short,arm_names=["grpo"])
        state=reload_experiment(path)
        c.ok("真实短训练重载一致",state["arms"]["grpo"]["greedy_ids_equal"])
        report=json.loads((path/"training.json").read_text(encoding="utf-8"))
        c.ok("短训练不含test选轮",report["no_test_driven_selection"])
        c.ok("短训练保存非空实际采样",(path/"grpo_rollouts.jsonl").stat().st_size>100)
    return {"count":len(c.names),"passed":c.names,
            "scope":"离线CPU数值与微型因果LM训练；非完整TRL/GRPO或大模型成绩"}

In [34]:
def mutation_checks() -> dict:
    detected=[]
    def catch(name,fn):
        try:fn()
        except (AssertionError,LessonError,ValueError,TypeError):
            detected.append({"fault":name,"detected":True});return
        raise AssertionError("故障未检出："+name)
    def wrong_a(data,cfg):
        r=reference_a(data,cfg)
        r["mask"]=torch.ones_like(r["mask"])
        return r
    catch("把EOS后PAD计入监督",lambda:check_exercise_a(wrong_a))
    def wrong_b(rewards,mode):
        val=rewards.detach()-rewards.detach().mean(1,keepdim=True)
        return val/(rewards.detach().std(1,keepdim=True,unbiased=True)+1e-6) if mode=="group" else val
    catch("使用不匹配的样本std",lambda:check_exercise_b(wrong_b))
    def wrong_c(new,old,mask,a,eps,mode,max_len):
        r=(new-old.detach()).exp()
        values=torch.maximum(r*a[:,None].detach(),r.clamp(1-eps,1+eps)*a[:,None].detach())
        values=values.masked_fill(~mask,0)
        if mode=="grpo":return -(values.sum(1)/mask.sum(1).clamp_min(1)).mean()
        if mode=="dapo":return -values.sum()/mask.sum()
        return -values.sum()/(new.shape[0]*max_len)
    catch("PPO min被错误替换为max",lambda:check_exercise_c(wrong_c))
    def wrong_length(new,old,mask,a,eps,mode,max_len):
        return token_clip_loss(new,old,mask,a,eps,"dapo",max_len)[0]
    catch("三种长度归约被悄悄合并",lambda:check_exercise_c(wrong_length))
    def wrong_reward():
        require(score_completion("02","horizon","02")["reward"]==1,
                "未EOS也被当正确完成")
    catch("把任务cap当正常EOS",wrong_reward)
    def wrong_external_bootstrap():
        r=torch.tensor([[0.,0.]],dtype=torch.float64)
        v=torch.tensor([[.2,.3]],dtype=torch.float64)
        nx=torch.tensor([[.3,.8]],dtype=torch.float64)
        mask=torch.tensor([[True,True]])
        term=torch.tensor([[False,True]]);trunc=torch.zeros_like(term)
        wrong=gae_with_boundaries(r,v,nx,mask,term,trunc,lam=1.)
        require(abs(float(wrong["return_target"][0,-1])-.76)<1e-8,
                "丢掉外部截断后的价值")
    catch("外部截断误当terminal",wrong_external_bootstrap)
    def wrong_gpu_estimate():
        estimate=kv_cache_bytes(32,8,2048,8,128,2)//2
        require(estimate==2*1024**3,"漏乘KV中V缓存")
    catch("只计算K不计算V",wrong_gpu_estimate)
    return {"count":len(detected),"cases":detected,
            "scope":"针对性的故障并非穷尽验证或安全认证"}

In [35]:
CHECK_RESULT=run_checks()
MUTATIONS=mutation_checks()
print('固定检查通过：',CHECK_RESULT['count'],'项；故障检出：',MUTATIONS['count'])
for case in MUTATIONS['cases']:
    print(' -',case['fault'])
if RUN_DIRECTORY is not None:
    save_json(RUN_DIRECTORY/'checks.json',CHECK_RESULT)
    save_json(RUN_DIRECTORY/'mutations.json',MUTATIONS)

训练token分支: grpo


结果目录: /tmp/tmpyzi9slw0/run_20261103_123503_069236Z


固定检查通过： 102 项；故障检出： 7
 - 把EOS后PAD计入监督
 - 使用不匹配的样本std
 - PPO min被错误替换为max
 - 三种长度归约被悄悄合并
 - 把任务cap当正常EOS
 - 外部截断误当terminal
 - 只计算K不计算V


## 19. 终端命令与验收

```bash
python lesson_66_llm_token_rlvr_grpo_engineering.py --checks-only
python lesson_66_llm_token_rlvr_grpo_engineering.py --math-only
python lesson_66_llm_token_rlvr_grpo_engineering.py --quick
python lesson_66_llm_token_rlvr_grpo_engineering.py
python lesson_66_llm_token_rlvr_grpo_engineering.py --arm dapo
python lesson_66_llm_token_rlvr_grpo_engineering.py --reload-run lesson_66_outputs/实际run目录
```

`--quick` 明确缩小预算，只是功能烟雾测试；不能把结果与完整训练当作相同实验。默认CPU，若系统有CUDA可明确指定 `--device cuda`，但当前制作环境没有CUDA，**不报告真实显存或加速收益**。

验收问题：为什么组内奖励全零不能说明模型正确？token 级ratio与整段ratio为什么不是同一PPO目标？长短回答的分母如何改变梯度权重？EOS和外部截断何时bootstrap？为什么ValueHead训练成功不等于GRPO已使用了critic？为什么4bit权重下界不能证明模型能装进3060？

**下一课第67课：从字符级RLVR走向真实Tokenizer、ChatTemplate与模型接口。** 将练习结果转换为可迁移的批次规范、真实EOS/stop处理和可用的GPU预算协议，在明确依赖与硬件边界后连接真实预训练模型。

## 主要来源与诚实边界

- [DeepSeekMath / GRPO](https://arxiv.org/abs/2402.03300)：组相对优势与基于规则奖励的背景。
- [TRL GRPO Trainer v0.28.0](https://huggingface.co/docs/trl/v0.28.0/en/grpo_trainer)：GRPO/DAPO/Dr. GRPO 损失归约、reward scaling、EOS与截断约定；版本固定，不声称未来所有实现相同。
- [TRL PPO](https://huggingface.co/docs/trl/main/ppo_trainer) 与 [PPO论文](https://arxiv.org/abs/1707.06347)：token级策略与价值模型的角色。
- [GAE论文](https://arxiv.org/abs/1506.02438) 与 [Gymnasium终止/截断](https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/)：外部截断bootstrap与任务终止的区别。
- [PyTorch attention mask语义](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention)：迁移到不同注意力API时的掩码约定不一定相同。
- [Hugging Face KV cache](https://huggingface.co/docs/transformers/kv_cache)：缓存大小、容量与offload的取舍。
- [PyTorch复现边界](https://docs.pytorch.org/docs/stable/notes/randomness.html)：seed固定不保证不同设备/软件版本逐位一致。

本课已在CPU执行完整自编模型与规则奖励，不曾下载/训练真实LLM；默认未使用GPU、量化、FlashAttention、vLLM、在线工具调用或数十亿参数模型。给出的显存数字是基于**声明假设的理论下界**，而不是任何具体3060的可运行承诺。

In [36]:
def main(argv=None) -> int:
    parser=argparse.ArgumentParser(description=__doc__)
    mode=parser.add_mutually_exclusive_group()
    mode.add_argument("--checks-only",action="store_true")
    mode.add_argument("--math-only",action="store_true")
    mode.add_argument("--quick",action="store_true")
    mode.add_argument("--reload-run",type=Path)
    parser.add_argument("--arm",choices=["grpo","dapo","dr_grpo","dr_noscale","all"],default="all")
    parser.add_argument("--device",choices=["cpu","cuda","auto"],default="cpu")
    parser.add_argument("--output-dir",type=Path,default=Path("lesson_66_outputs"))
    args=parser.parse_args(argv)
    torch.set_num_threads(1)
    if args.reload_run:
        print(json.dumps(reload_experiment(args.reload_run,args.device),ensure_ascii=False,indent=2));return 0
    report=run_checks()
    print(f"固定检查通过：{report['count']}项；仅离线微型LLM/token级RLVR机制。")
    if args.checks_only:return 0
    if args.math_only:
        folder=args.output_dir/("math_"+datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%fZ"))
        folder.mkdir(parents=True,exist_ok=False)
        save_json(folder/"mechanisms.json",{"memory":memory_lower_bounds(Config()),
                 "mutations":mutation_checks(),"checks":report})
        print("输出目录:",folder.resolve());return 0
    cfg=Config(sft_steps=4,updates=3,critic_steps=6) if args.quick else Config()
    arms=None if args.arm=="all" else [args.arm]
    folder=full_experiment(args.output_dir,cfg,args.device,arms)
    save_json(folder/"checks.json",report)
    save_json(folder/"mutations.json",mutation_checks())
    print("完整实验及检查已保存:",folder.resolve())
    return 0